# Mega Project 1 — Intelligent Underwriting & Automated Credit Decisioning
## Problem 3: Loan Application Approval — EDA, Multi-Table Feature Engineering,
## Top-4 Model Screening → Top-2 5-Fold CV Champion Selection, SHAP + LIME
## Explainability, Statistical Validation & Financial-Impact Reporting

**Home Credit Default Risk — 6 Mega Projects Enterprise Suite**

### Business context
Automate the green-light / decline decision for incoming credit requests, using
Home Credit's own real history of past application decisions as the training
signal — the same real outcomes underwriters actually produced.

### Data used (real, verified against your files before this notebook was written) —
### 7 real raw files integrated, up from 2 in the original version of this notebook
- `previous_application.csv` — 1,670,214 real past applications. Real
  `NAME_CONTRACT_STATUS` distribution: 1,036,781 Approved / 316,319 Canceled /
  290,678 Refused / 26,436 Unused offer.
- `application_train.csv` — joined in for applicant-profile context (income,
  age, education, family status, EXT_SOURCE_2).
- `bureau.csv` + `bureau_balance.csv` — real external credit-bureau history,
  aggregated to applicant level (used in full — no linkage to any specific Home
  Credit decision, so no leakage risk regardless of target granularity).
- `POS_CASH_balance.csv` + `installments_payments.csv` + `credit_card_balance.csv`
  — real prior-loan servicing history, aggregated to applicant level with an
  **exact leave-one-out (LOO) join**: each row only ever sees that same
  applicant's *other* previous applications, never its own — because these 3
  tables only contain records for loans that were actually disbursed, and would
  otherwise leak the very approval outcome this notebook predicts.

### Hardware-utilization fix (this revision — real root cause, not a hardware limit)
A prior real run of this notebook was observed using far less of the CPU/RAM
ceiling it was configured for. Root cause, found by inspecting the actual code
order: the thread-count environment variables (`OMP_NUM_THREADS`,
`OPENBLAS_NUM_THREADS`, `MKL_NUM_THREADS`, `POLARS_MAX_THREADS`, etc.) were
never actually being *set* anywhere, **and** numpy/polars/pandas/scikit-learn/
xgboost/lightgbm/catboost were all imported at the very top of the file, before
the ceiling was even computed — every one of those libraries reads its own
thread-count env var exactly once, at its own import time, so no library ever
saw a real ceiling regardless of what was printed. Combined with
`GradientBoostingClassifier` (no multi-core support at all in scikit-learn) and
`LogisticRegression` (whose `n_jobs` has no effect for binary classification)
sitting in the old 6-model benchmark, this is the concrete, code-level
explanation — not a hardware limitation. Fixed this revision by:
- Importing and calling the shared `src/utils/performance_setup.py` module
  (HYPER) as literally the first executable step, before any BLAS/OpenMP-
  reading library is imported — env vars are now actually set before they are
  read.
- Explicitly pinning this process's CPU affinity to every detected logical
  core, removing any pre-existing OS-level core restriction the env vars alone
  cannot fix.
- Dropping GradientBoostingClassifier and LogisticRegression from the model
  set (see below) — both would otherwise run pinned to one thread while every
  other configured thread sits idle.
- Adding Parquet-over-CSV caching (WARP): the first real run still pays the
  normal CSV parse cost, but every run after that reads a much faster cached
  columnar Parquet copy of each of the 7 raw files instead — a genuine, stated
  trade-off, not a claim that the first run gets faster too.

### Model benchmark redesign: Top-4 screen → Top-2 5-fold CV (reduced from 6 models)
1. **Stage A — screening**: all 4 real candidates (RandomForest, XGBoost,
   CatBoost, LightGBM — chosen for real predictive strength on tabular credit
   data *and* genuine multi-core parallelism) are each fit once on an 80/20
   real train/validation split.
2. **Stage B — champion selection**: only the top 2 screened candidates are
   promoted to a real 5-fold `StratifiedKFold` CV (roughly half the CV cost of
   the old 6-model × 5-fold benchmark). The champion is the higher mean-CV-AUC
   model of those two.
3. **CV Report**: a real per-fold ROC-AUC table + chart for the top-2 models,
   not just their mean/std.
4. **SHAP explainability** (champion model only): `shap.TreeExplainer` on a
   real 300-row holdout sample — a mean-|SHAP| bar chart plus a beeswarm detail
   chart.
5. **LIME explainability** (champion model only): real local explanations for
   3 representative real holdout cases (most-confident correct approval,
   most-confident correct decline, a real misclassified application).

### New this revision: real cross-problem feature — PD from Notebook 01 (soft dependency)
This notebook's model now genuinely consumes Notebook 01's real, independently-
trained default-risk model as a new real input feature. If Notebook 01's
champion model artifact is present, this notebook rebuilds Notebook 01's exact
customer-level feature set via the shared `src/features/credit_default_features.py`
module (reusing the same real `application_train`/`bureau` data already loaded
above — no extra file read), scores each real customer's current
probability-of-default once, and joins that one real PD onto every in-scope
row here by `SK_ID_CURR` as `UPSTREAM_PD_FROM_NB01` — a real numeric feature
the top-4 screen, top-2 CV, champion, and SHAP/LIME explainability all
genuinely see and use, not a decorative addition. **Explicit, disclosed
temporal-snapshot caveat**: this PD reflects each customer's CURRENT risk
profile, not a reconstruction of their risk at the actual historical moment
of each specific previous-application decision this notebook predicts — a
real, stated limitation, not leakage of this notebook's own TARGET (a
different real outcome column from Notebook 01's). If Notebook 01 has not
been run yet, this feature is skipped and this notebook trains standalone
exactly as before — a soft dependency, not a hard requirement.

### Scope (explicit, reported — not silently applied)
This notebook covers previous applications whose customer also appears in
`application_train` (the join population available to this notebook). The exact
in-scope row count and percentage are computed and printed live, not assumed.

### Leakage guards (two, both enforced at runtime, not just documented)
1. Every `previous_application` column only populated *after* a decision is made —
   `CODE_REJECT_REASON`, the five `DAYS_FIRST_DRAWING` / `DAYS_FIRST_DUE` /
   `DAYS_LAST_DUE_1ST_VERSION` / `DAYS_LAST_DUE` / `DAYS_TERMINATION`, and
   `NFLAG_INSURED_ON_APPROVAL` — is explicitly excluded from the feature set and
   asserted against at runtime.
2. The new POS_CASH/installments/credit_card applicant-level features are computed
   as TOTAL minus this-row's-OWN contribution (leave-one-out), verified >= 0 for
   every row by an integrity self-check before any reporting is generated.

### What this notebook does (SOP Stages 1B–6 in one run)
1. Applies the WARP hardware fix above, then loads all 7 real files via a
   Parquet-cached Polars read.
2. Runs real Exploratory Data Analysis & data-quality checks on the in-scope data
   *before* feature engineering — missingness, IQR outliers, target correlation —
   with 3 vivid multicolor chart figures (SOP Stage 1B/2).
3. Engineers 18 real applicant credit-history features from the 5 supplementary
   tables via the shared `src/features/applicant_credit_history_features.py`
   module (HYPER standard), on top of the original 29 request/applicant-level
   features — 47 features total (48 when Notebook 01's real PD feature is
   also included, see above).
4. Cross-notebook feature: scores and joins Notebook 01's real PD as
   `UPSTREAM_PD_FROM_NB01`, if available (see above).
5. Splits train/holdout (85/15, stratified, seed 42) — encoders and imputers are
   fit on train only, never on holdout (no leakage).
6. Runs the Top-4 screen → Top-2 5-fold CV champion selection described above;
   retrains the champion on full train, evaluates once on the untouched holdout
   set; displays the ROC curve, screening chart, CV benchmark chart, and CV
   report chart inline (vivid multicolor, per the standing chart-style rule).
7. Computes and displays real SHAP + LIME explainability for the champion model
   — including `UPSTREAM_PD_FROM_NB01`'s real measured contribution, when present.
8. Runs real Statistical Validation (SOP Stage 4): bootstrap 95% CI on holdout
   ROC-AUC, calibration-by-decile, split-half PSI, and an explicit deployment
   readiness verdict.
9. Runs 16-18 integrity self-checks (the extra 2 only when Notebook 01's PD
   feature is integrated; fails loudly rather than silently passing bad
   state — including checks that the CPU thread ceiling was actually
   applied, the screen was really top-4, CV really ran on only 2 models, SHAP
   values are finite, and LIME explanations were really computed), then
   generates a full Stage-5 reporting package — CSV outputs (including model
   screening, CV report, SHAP importances, and LIME explanations as their own
   files), a colorized Word report, a 12-sheet Excel workbook, and an HTML
   dashboard with 8 live charts (4 with slicer/filter dropdowns) — via the
   shared `src/reporting/report_builder.py` module (HYPER).
10. Saves the champion model + a full run summary (including the real
    performance-config, model-selection, explainability, and interdependency
    metadata) to `../decision_engine/artifacts/` — overwritten in place every
    run (idempotent), never appended.

### Standing rules this notebook follows
- **Zero-fabrication**: every number below is computed live, this run, from your own
  copy of the real data — nothing is carried over from a prior session.
- **WARP**: resource ceilings capped at 90% RAM / 95% CPU threads (never 100%,
  a safety ceiling — not a floor forced by padding), applied *before* any
  heavy import; CPU affinity pinned; Parquet-over-CSV caching; vectorized
  Polars throughout; `RANDOM_SEED = 42`.
- **HYPER**: shared `src/features/`, `src/reporting/`, and `src/utils/` modules,
  built once, imported here and reusable by any future notebook.

### Before you run this
Uses the same `project_config.json` as Notebook 01 (project root, `raw_data_dir`
pointing at your real Kaggle CSV folder — must contain all 7 files listed above).
Runs standalone even without Notebook 01 (soft dependency) — run Notebook 01
first if you want the real `UPSTREAM_PD_FROM_NB01` feature included. On your
real machine, `shap` and `lime` must be installed (`pip install shap lime`) —
both are lightweight, pure-Python-plus-compiled-extension packages with no
extra system dependency.

### Verification status
Verified end-to-end on a synthetic fixture matching the real schema via real
Jupyter execution (`jupyter nbconvert --execute`), in BOTH states — with and
without Notebook 01's model artifact present — 0 errors either way, all
integrity checks (16 without Notebook 01, 18 with it) passed. HTML dashboard
confirmed rendering 8 charts / 4 filter dropdowns with 0 console errors under
a network-blocked Playwright check, Excel formulas confirmed correct via
LibreOffice headless recalculation, and the Parquet cache confirmed correct (a
second read returns byte-identical data to the original CSV read). **Not yet
run against your real data.**


In [ ]:
# ============================================================================
# NOTEBOOK 02 — MEGA PROJECT 3: INTELLIGENT UNDERWRITING & AUTOMATED CREDIT
# DECISIONING | PROBLEM 3: LOAN APPLICATION APPROVAL
# Business Understanding, EDA, Multi-Table Feature Engineering, Top-4 Model
# Screening -> Top-2 5-Fold CV Champion Selection, SHAP + LIME Explainability,
# Statistical Validation & Financial-Impact Reporting (SOP Stages 1-6 in one run)
# ----------------------------------------------------------------------------
# Zero-fabrication notice: every number this cell prints, plots, or writes to a
# report is computed live, right now, from YOUR OWN copy of the real Kaggle Home
# Credit Default Risk dataset. Nothing here is carried over from a prior run or
# invented. Re-running this cell always overwrites the same output paths
# (idempotent).
#
# HARDWARE-UTILIZATION FIX (this revision): every BLAS/OpenMP thread-count
# environment variable is now set — via the shared, HYPER src/utils/
# performance_setup.py module, not a local duplicate — BEFORE numpy, polars,
# pandas, scikit-learn, xgboost, lightgbm, or catboost are imported anywhere
# below. The PREVIOUS version of this file computed a thread ceiling but never
# actually set any environment variable, AND it imported all of those libraries
# at the very top of the file, before that ceiling was even computed — so no
# library ever saw a real ceiling regardless. Combined with GradientBoosting
# Classifier (no multi-core support at all in scikit-learn) and LogisticRegres-
# sion (n_jobs has no effect for binary classification) sitting in the old
# 6-model benchmark, this is the concrete, code-level explanation for a real
# run observably using a fraction of the CPU/RAM ceiling it was configured for
# — not a hardware limitation. See PERFORMANCE_SETUP_README.md / WARP notes.
# ============================================================================

import os
import sys
import json
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# SECTION 1 — Config + suite-root resolution (stdlib only — no heavy library
# is imported yet, deliberately, so the WARP thread ceiling below can be set
# before any of them read their thread-count environment variables. Never
# print the resolved raw-data path itself: this notebook may be shared
# publicly, e.g. on GitHub/Kaggle).
# ---------------------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd()
def _find_suite_root(start: Path = None) -> Path:
    """Locate the home-credit-enterprise-suite project root (the folder containing
    project_config.json), regardless of where this notebook's kernel actually launched
    from. Checked in order, fastest and most explicit first -- deliberately NOT an
    unbounded/recursive filesystem scan (the exact "hangs / looks frozen" risk this
    suite's WARP performance module exists to avoid):
    1. HC_SUITE_ROOT environment variable, if set (see PERFORMANCE_SETUP_README.md)
    2. Walking UPWARD from the working directory (covers: cwd is this notebook's own
       mega_project_.../notebooks/ folder, the normal case when opened in place)
    3. A short list of well-known locations under the home directory (covers: the
       working directory being your home folder itself -- an ANCESTOR of the project,
       not inside it -- which an upward-only search cannot reach)
    """
    start = start or Path.cwd()
    marker = "project_config.json"
    env_override = os.environ.get("HC_SUITE_ROOT")
    if env_override and (Path(env_override) / marker).exists():
        return Path(env_override)
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate
    for candidate in [
        Path.home() / "Downloads" / "home-credit-enterprise-suite",
        Path.home() / "home-credit-enterprise-suite",
        Path.home() / "Desktop" / "home-credit-enterprise-suite",
        start / "home-credit-enterprise-suite",
        start / "Downloads" / "home-credit-enterprise-suite",
    ]:
        if (candidate / marker).exists():
            return candidate
    return None


SUITE_ROOT = _find_suite_root()
if SUITE_ROOT is None:
    raise FileNotFoundError(
        "project_config.json not found. Checked upward from the working directory plus "
        "well-known locations under your home folder. Fix: either open this notebook's "
        "own .ipynb file in place (rather than running its code in a fresh kernel "
        "elsewhere), or set an environment variable before launching Jupyter, e.g. on "
        'Windows PowerShell: $env:HC_SUITE_ROOT="C:\\Users\\rnand\\Downloads\\'
        'home-credit-enterprise-suite" -- see PERFORMANCE_SETUP_README.md.'
    )
config_path = SUITE_ROOT / "project_config.json"
with open(config_path) as f:
    CONFIG = json.load(f)

RAW_DIR = Path(CONFIG["raw_data_dir"])
SEED = int(CONFIG.get("random_seed", 42))
RANDOM_SEED = SEED

ARTIFACTS_DIR = SUITE_ROOT / "01_mega_project_1_underwriting_approval" / "decision_engine" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = SUITE_ROOT / "01_mega_project_1_underwriting_approval" / "decision_engine" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_CACHE_DIR = SUITE_ROOT / "01_mega_project_1_underwriting_approval" / "decision_engine" / "_parquet_cache"

# HYPER standing rule: shared feature-engineering + reporting + performance logic
# lives in src/, imported here rather than duplicated inline.
sys.path.insert(0, str(SUITE_ROOT / "src"))
from utils.performance_setup import (
    configure_performance, pin_cpu_affinity, sklearn_n_jobs, gbm_thread_kwargs,
    threadpool_guard, free_memory, check_ram_headroom, load_csv_cached,
)

# ---------------------------------------------------------------------------
# SECTION 2 — WARP resource ceilings (hard cap, never 100%) — set BEFORE any
# of numpy/polars/pandas/scikit-learn/xgboost/lightgbm/catboost are imported.
# configure_performance() sets OMP_NUM_THREADS / OPENBLAS_NUM_THREADS /
# MKL_NUM_THREADS / NUMEXPR_NUM_THREADS / POLARS_MAX_THREADS (etc.) as real OS
# environment variables — every one of those libraries reads its own copy of
# these exactly once, at its own import/init time, so this call MUST happen
# first. pin_cpu_affinity() additionally pins this process to every detected
# logical core, removing any pre-existing OS-level core restriction the env
# vars alone cannot fix.
# ---------------------------------------------------------------------------
PERF = configure_performance(
    ram_ceiling_fraction=float(CONFIG.get("ram_ceiling_fraction", 0.90)),
    cpu_ceiling_fraction=float(CONFIG.get("cpu_ceiling_fraction", 0.95)),
)
pin_cpu_affinity(PERF)
TOTAL_RAM_GB = PERF["total_ram_gb"]
TOTAL_THREADS = PERF["logical_cores"]
RAM_CEILING_GB = PERF["ram_ceiling_gb"]
CPU_CEILING_THREADS = PERF["n_threads"]

# ---------------------------------------------------------------------------
# SECTION 3 — Heavy-library imports (deliberately AFTER Section 2 above)
# ---------------------------------------------------------------------------
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score, brier_score_loss,
    roc_curve, confusion_matrix,
)
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
import pandas as pd
import shap
import joblib
from lime.lime_tabular import LimeTabularExplainer

np.random.seed(SEED)
T0 = time.time()

from features.applicant_credit_history_features import (
    engineer_bureau_history_features, engineer_prev_loan_servicing_features_loo,
)
from features.credit_default_features import engineer_credit_default_features
from reporting.report_builder import (
    write_csv_outputs, build_word_report, build_excel_workbook,
    build_html_dashboard, assumption_ref, VIVID_PALETTE, _palette,
)
# Standing chart-style rule (all problems): vivid multicoloured charts everywhere,
# using the 8-hue CVD-validated categorical palette canonicalized in
# src/reporting/report_builder.py (HYPER -- imported, not redefined per notebook).

# 7 real raw files now integrated (up from 2): previous_application + application_train
# for scope/target/applicant-profile, plus bureau + bureau_balance + POS_CASH_balance +
# installments_payments + credit_card_balance for a materially richer applicant
# credit-history feature set (more real feature columns -> higher real model accuracy).
REQUIRED_FILES = [
    "previous_application.csv", "application_train.csv", "bureau.csv", "bureau_balance.csv",
    "POS_CASH_balance.csv", "installments_payments.csv", "credit_card_balance.csv",
]
missing = [fn for fn in REQUIRED_FILES if not (RAW_DIR / fn).exists()]
if missing:
    raise FileNotFoundError(f"Missing required raw file(s) in configured raw_data_dir: {missing}")

print(f"[WARP] {TOTAL_RAM_GB:.1f} GB RAM / {TOTAL_THREADS} threads detected -> "
      f"ceiling {RAM_CEILING_GB} GB RAM, {CPU_CEILING_THREADS} threads "
      f"(env vars applied before any heavy import; CPU affinity pinned to all cores)")
print("[DATA] Raw data directory resolved and verified (path withheld from output by design).")
print(f"[SEED] RANDOM_SEED = {SEED} (fixed for full reproducibility)")

# ---------------------------------------------------------------------------
# SECTION 4 — Load real data (WARP: Parquet-over-CSV cache — first run reads
# the real CSV and writes a Parquet cache under decision_engine/_parquet_cache/;
# every run after that reads the much-faster columnar Parquet copy instead.
# Never mutates your raw Kaggle folder.)
# ---------------------------------------------------------------------------
prev = load_csv_cached(RAW_DIR / "previous_application.csv", PARQUET_CACHE_DIR, null_values=["", "NA", "XNA", "XAP"])
app = load_csv_cached(RAW_DIR / "application_train.csv", PARQUET_CACHE_DIR, null_values=["", "NA", "XNA"])
bureau = load_csv_cached(RAW_DIR / "bureau.csv", PARQUET_CACHE_DIR, null_values=["", "NA", "XNA"])
bureau_balance = load_csv_cached(RAW_DIR / "bureau_balance.csv", PARQUET_CACHE_DIR, null_values=["", "NA"])
pos_cash = load_csv_cached(RAW_DIR / "POS_CASH_balance.csv", PARQUET_CACHE_DIR, null_values=["", "NA", "XNA"])
installments = load_csv_cached(RAW_DIR / "installments_payments.csv", PARQUET_CACHE_DIR, null_values=["", "NA"])
credit_card = load_csv_cached(RAW_DIR / "credit_card_balance.csv", PARQUET_CACHE_DIR, null_values=["", "NA", "XNA"])
check_ram_headroom(PERF)

N_PREV_RAW = prev.height
print(f"[LOAD] previous_application: {N_PREV_RAW:,} rows x {prev.width} cols")
print(f"[LOAD] application_train (applicant-profile join): {app.height:,} rows")
print(f"[LOAD] bureau: {bureau.height:,} rows | bureau_balance: {bureau_balance.height:,} rows")
print(f"[LOAD] POS_CASH_balance: {pos_cash.height:,} rows | installments_payments: {installments.height:,} rows "
      f"| credit_card_balance: {credit_card.height:,} rows")

if "NAME_CONTRACT_STATUS" not in prev.columns:
    raise ValueError("NAME_CONTRACT_STATUS column missing from previous_application.csv.")

# ---------------------------------------------------------------------------
# SECTION 4 — Scope & target definition (explicit ASSUMPTION, source-noted)
# ---------------------------------------------------------------------------
# ASSUMPTION: restricted to previous applications whose customer also appears in
# application_train (the join population this notebook has available). This is a
# real, computed subset -- not the full previous_application population -- and is
# reported explicitly below rather than silently applied.
status_counts_full = prev.group_by("NAME_CONTRACT_STATUS").agg(pl.len().alias("n")).sort("n", descending=True)
print("[EDA] Real NAME_CONTRACT_STATUS distribution (full previous_application):")
print(status_counts_full)

prev = prev.with_columns((pl.col("NAME_CONTRACT_STATUS") == "Approved").cast(pl.Int8).alias("TARGET"))

# leakage guard: exclude every column only populated post-decision for approved loans,
# and the reject-reason field (a direct restatement of the target for Refused rows)
LEAKAGE_COLS = {
    "CODE_REJECT_REASON", "DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE",
    "DAYS_LAST_DUE_1ST_VERSION", "DAYS_LAST_DUE", "DAYS_TERMINATION",
    "NFLAG_INSURED_ON_APPROVAL", "NAME_CONTRACT_STATUS",
}

applicant_cols = ["SK_ID_CURR", "AMT_INCOME_TOTAL", "DAYS_BIRTH", "NAME_EDUCATION_TYPE",
                  "NAME_FAMILY_STATUS", "OCCUPATION_TYPE", "EXT_SOURCE_2"]
applicant_cols = [c for c in applicant_cols if c in app.columns]
df = prev.join(app.select(applicant_cols), on="SK_ID_CURR", how="inner")
N_JOINED = df.height
join_rate = N_JOINED / N_PREV_RAW if N_PREV_RAW else 0.0
print(f"[SCOPE] {N_JOINED:,} / {N_PREV_RAW:,} previous-application rows "
      f"({join_rate:.2%}) have a matching applicant profile in application_train "
      f"and are in scope for this notebook.")

df = df.with_columns((-pl.col("DAYS_BIRTH") / 365.25).alias("APPLICANT_AGE_YEARS"))

# ---------------------------------------------------------------------------
# SECTION 5 — Exploratory Data Analysis & Data Quality (SOP Stage 1B/2)
# Real EDA computed on the in-scope, joined previous-application data, before any
# feature engineering below -- every chart shows what your actual data looks like
# going into this notebook, not a post-engineering view dressed up as "before".
# ---------------------------------------------------------------------------
N_DF = df.height
null_counts = df.null_count().to_pandas().T.reset_index()
null_counts.columns = ["column", "n_null"]
null_counts["pct_null"] = null_counts["n_null"] / N_DF
null_counts = null_counts[null_counts["n_null"] > 0].sort_values("pct_null", ascending=False)
top_missing = null_counts.head(15)
print(f"[EDA] {len(null_counts)} / {df.width} real in-scope columns have at least one missing value. "
      f"Top 5 by % missing:")
for _, row in null_counts.head(5).iterrows():
    print(f"  {row['column']}: {row['pct_null']:.2%} ({int(row['n_null']):,} rows)")

target_counts = df["TARGET"].value_counts().sort("TARGET").to_pandas()
APPROVAL_RATE = float(df["TARGET"].mean())
print(f"[EDA] Real approval rate in scope: {APPROVAL_RATE:.4%}")

contract_type_counts = df["NAME_CONTRACT_TYPE"].drop_nulls().value_counts().sort("count", descending=True).to_pandas()
goods_category_counts = df["NAME_GOODS_CATEGORY"].drop_nulls().value_counts().sort("count", descending=True).to_pandas()
contract_type_counts["NAME_CONTRACT_TYPE"] = contract_type_counts["NAME_CONTRACT_TYPE"].astype(str)
goods_category_counts["NAME_GOODS_CATEGORY"] = goods_category_counts["NAME_GOODS_CATEGORY"].astype(str)

DIST_COLS = [c for c in ["AMT_APPLICATION", "AMT_CREDIT", "AMT_ANNUITY"] if c in df.columns]
dist_data = {c: df[c].drop_nulls().to_numpy() for c in DIST_COLS}

OUTLIER_SUMMARY = []
for c in DIST_COLS:
    vals = dist_data[c]
    q1, q3 = np.percentile(vals, [25, 75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((vals < lo) | (vals > hi)).sum())
    OUTLIER_SUMMARY.append({"column": c, "n_outliers_iqr": n_out, "pct_outliers_iqr": n_out / len(vals),
                             "iqr_lower": float(lo), "iqr_upper": float(hi)})
    print(f"[EDA] IQR outliers in {c}: {n_out:,} ({n_out / len(vals):.2%}) outside [{lo:,.0f}, {hi:,.0f}]")

CORR_COLS = [c for c in [
    "AMT_ANNUITY", "AMT_APPLICATION", "AMT_CREDIT", "AMT_DOWN_PAYMENT", "AMT_GOODS_PRICE",
    "RATE_DOWN_PAYMENT", "SELLERPLACE_AREA", "CNT_PAYMENT", "DAYS_DECISION",
    "AMT_INCOME_TOTAL", "APPLICANT_AGE_YEARS", "EXT_SOURCE_2",
] if c in df.columns]
corr_pdf = df.select(["TARGET"] + CORR_COLS).to_pandas()
target_corr = corr_pdf.corr(numeric_only=True)["TARGET"].drop("TARGET").sort_values()
print(f"[EDA] Real correlation with TARGET (top 3 by |r|): "
      f"{target_corr.abs().sort_values(ascending=False).head(3).round(4).to_dict()}")

# --- EDA Figure 1: Data Quality Overview (2x2, vivid multicolor) -----------
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

axes[0, 0].barh(top_missing["column"][::-1], (top_missing["pct_null"][::-1] * 100),
                 color=_palette(len(top_missing)))
axes[0, 0].set_xlabel("% Missing"); axes[0, 0].set_title(f"Top {len(top_missing)} Columns by Real Missing-Value %")

axes[0, 1].bar(["Not Approved (0)", "Approved (1)"], target_counts["count"].tolist(),
                color=["#d03b3b", "#0ca30c"])  # status palette: critical / good
axes[0, 1].set_title(f"Real Approval Outcome Balance (approval rate {APPROVAL_RATE:.2%})")
for i, v in enumerate(target_counts["count"].tolist()):
    axes[0, 1].text(i, v, f"{v:,}", ha="center", va="bottom")

top_contract_types = contract_type_counts.head(6)
axes[1, 0].bar(top_contract_types["NAME_CONTRACT_TYPE"], top_contract_types["count"],
                color=_palette(len(top_contract_types)))
axes[1, 0].set_title("Real NAME_CONTRACT_TYPE Distribution (in scope)")
axes[1, 0].tick_params(axis="x", rotation=20)

top_goods = goods_category_counts.head(6)
axes[1, 1].bar(top_goods["NAME_GOODS_CATEGORY"], top_goods["count"], color=_palette(len(top_goods)))
axes[1, 1].set_title("Real NAME_GOODS_CATEGORY Distribution (top 6, in scope)")
axes[1, 1].tick_params(axis="x", rotation=30)

plt.tight_layout()
eda_overview_path = REPORTS_DIR / "notebook_02_eda_overview.png"
plt.savefig(eda_overview_path, dpi=110)
plt.show()

# --- EDA Figure 2: Numeric Distributions & IQR Outliers (2xN, vivid) -------
fig, axes = plt.subplots(2, len(DIST_COLS), figsize=(5 * len(DIST_COLS), 8))
for i, c in enumerate(DIST_COLS):
    axes[0, i].hist(dist_data[c], bins=40, color=VIVID_PALETTE[i % len(VIVID_PALETTE)], edgecolor="white")
    axes[0, i].set_title(f"Real Distribution: {c}")
    bx = axes[1, i].boxplot(dist_data[c], vert=False, patch_artist=True)
    bx["boxes"][0].set_facecolor(VIVID_PALETTE[(i + 3) % len(VIVID_PALETTE)])
    n_out = next(o["n_outliers_iqr"] for o in OUTLIER_SUMMARY if o["column"] == c)
    axes[1, i].set_title(f"IQR Outliers: {n_out:,} ({n_out / len(dist_data[c]):.1%})")
plt.tight_layout()
eda_distributions_path = REPORTS_DIR / "notebook_02_eda_distributions.png"
plt.savefig(eda_distributions_path, dpi=110)
plt.show()

# --- EDA Figure 3: Real correlation with TARGET (sign-colored, vivid) ------
fig, ax = plt.subplots(figsize=(9, 5.5))
colors = [VIVID_PALETTE[0] if v >= 0 else VIVID_PALETTE[7] for v in target_corr]
ax.barh(target_corr.index, target_corr.values, color=colors)
ax.axvline(0, color="#898781", linewidth=1)
ax.set_title("Real Pearson Correlation with Approval TARGET\n(blue = positive, red = negative)")
plt.tight_layout()
eda_correlation_path = REPORTS_DIR / "notebook_02_eda_correlation.png"
plt.savefig(eda_correlation_path, dpi=110)
plt.show()

EDA_CHART_PATHS = [eda_overview_path, eda_distributions_path, eda_correlation_path]

# ---------------------------------------------------------------------------
# SECTION 6 — Multi-table applicant credit-history feature engineering (WARP:
# vectorized Polars aggregation throughout) via the shared src/features module
# (HYPER: built once, reusable by any future notebook needing the same tables).
#
# Leakage guard for the 3 SK_ID_PREV-linked tables (POS_CASH_balance,
# installments_payments, credit_card_balance): each only contains servicing
# records for loans that were actually DISBURSED. Naively aggregating them at
# SK_ID_CURR level would let a row's own (or a same-applicant sibling row's)
# post-approval servicing history leak into its own approval-prediction feature.
# Fixed with an exact leave-one-out (LOO) aggregate: TOTAL (every servicing
# record for this applicant) minus OWN (just this specific SK_ID_PREV's own
# records) = every OTHER previous application this same applicant has, which
# carries zero information about the specific row being scored.
# bureau / bureau_balance carry no SK_ID_PREV linkage at all (external credit
# bureau data, independent of Home Credit's own decisions) so they are used in
# full, at SK_ID_CURR level -- the same accepted convention as Notebook 01.
# ---------------------------------------------------------------------------
bureau_feats_df, BUREAU_FEATURES = engineer_bureau_history_features(bureau, bureau_balance)
servicing_totals_df, servicing_own_df, SERVICING_FEATURES = engineer_prev_loan_servicing_features_loo(
    pos_cash, installments, credit_card
)

df = df.join(bureau_feats_df, on="SK_ID_CURR", how="left")
df = df.with_columns([pl.col(c).fill_null(0) for c in BUREAU_FEATURES])

df = df.join(servicing_totals_df, on="SK_ID_CURR", how="left")
df = df.join(servicing_own_df, on="SK_ID_PREV", how="left")
_tot_own_cols = [c for c in servicing_totals_df.columns if c != "SK_ID_CURR"] + \
                [c for c in servicing_own_df.columns if c != "SK_ID_PREV"]
df = df.with_columns([pl.col(c).fill_null(0) for c in _tot_own_cols])

df = df.with_columns([
    (pl.col("POS_N_TOT") - pl.col("POS_N_OWN")).clip(lower_bound=0).alias("POS_CNT_RECORDS"),
    (pl.col("POS_N_COMPLETED_TOT") - pl.col("POS_N_COMPLETED_OWN")).clip(lower_bound=0).alias("POS_CNT_COMPLETED"),
    (pl.col("POS_SUM_SK_DPD_DEF_TOT") - pl.col("POS_SUM_SK_DPD_DEF_OWN")).clip(lower_bound=0).alias("_pos_dpd_def_loo"),
    (pl.col("INSTAL_N_TOT") - pl.col("INSTAL_N_OWN")).clip(lower_bound=0).alias("INSTAL_CNT_PAYMENTS"),
    (pl.col("INSTAL_SUM_PAY_RATIO_TOT") - pl.col("INSTAL_SUM_PAY_RATIO_OWN")).alias("_instal_sum_pay_ratio_loo"),
    (pl.col("INSTAL_N_LATE_TOT") - pl.col("INSTAL_N_LATE_OWN")).clip(lower_bound=0).alias("_instal_n_late_loo"),
    (pl.col("INSTAL_SUM_DAYS_LATE_TOT") - pl.col("INSTAL_SUM_DAYS_LATE_OWN")).clip(lower_bound=0).alias("_instal_sum_days_late_loo"),
    (pl.col("CC_N_TOT") - pl.col("CC_N_OWN")).clip(lower_bound=0).alias("CC_CNT_RECORDS"),
    (pl.col("CC_SUM_UTILIZATION_TOT") - pl.col("CC_SUM_UTILIZATION_OWN")).clip(lower_bound=0).alias("_cc_sum_util_loo"),
    (pl.col("CC_SUM_BALANCE_TOT") - pl.col("CC_SUM_BALANCE_OWN")).clip(lower_bound=0).alias("_cc_sum_balance_loo"),
    (pl.col("CC_SUM_SK_DPD_TOT") - pl.col("CC_SUM_SK_DPD_OWN")).clip(lower_bound=0).alias("CC_SUM_SK_DPD"),
])
df = df.with_columns([
    (pl.col("_pos_dpd_def_loo") / (pl.col("POS_CNT_RECORDS") + 1.0)).alias("POS_MEAN_SK_DPD_DEF"),
    (pl.col("_instal_sum_pay_ratio_loo") / (pl.col("INSTAL_CNT_PAYMENTS") + 1.0)).alias("INSTAL_MEAN_PAYMENT_RATIO"),
    (pl.col("_instal_n_late_loo") / (pl.col("INSTAL_CNT_PAYMENTS") + 1.0)).alias("INSTAL_PCT_LATE"),
    (pl.col("_instal_sum_days_late_loo") / (pl.col("_instal_n_late_loo") + 1.0)).alias("INSTAL_MEAN_DAYS_LATE_WHEN_LATE"),
    (pl.col("_cc_sum_util_loo") / (pl.col("CC_CNT_RECORDS") + 1.0)).alias("CC_MEAN_UTILIZATION"),
    (pl.col("_cc_sum_balance_loo") / (pl.col("CC_CNT_RECORDS") + 1.0)).alias("CC_MEAN_BALANCE"),
])

CREDIT_HISTORY_FEATURES = BUREAU_FEATURES + SERVICING_FEATURES
print(f"[FEATURES] Multi-table integration added {len(CREDIT_HISTORY_FEATURES)} real applicant "
      f"credit-history features ({len(BUREAU_FEATURES)} from bureau/bureau_balance, "
      f"{len(SERVICING_FEATURES)} leave-one-out-safe from POS_CASH/installments/credit_card).")

# ---------------------------------------------------------------------------
# SECTION 6B — Real cross-notebook feature: PD from Notebook 01 (soft
# dependency, genuine model INPUT feature this time, not just a reporting
# cross-check like Notebooks 04/05 -- this notebook's model actually trains
# on it when it is available).
#
# If Notebook 01's champion model artifact is present, this notebook rebuilds
# Notebook 01's EXACT customer-level feature set via the shared feature
# module (reusing the same real `app` / `bureau` Polars frames already
# loaded in Section 4 above -- no extra file read), scores each real
# customer's CURRENT probability of default once, and joins that one real PD
# onto every in-scope row of THIS notebook by SK_ID_CURR as a new real
# numeric input feature, UPSTREAM_PD_FROM_NB01 -- genuine cross-problem
# interdependency, not a forced or fabricated one.
#
# TEMPORAL-SNAPSHOT CAVEAT (explicit, disclosed, not hidden): this PD
# reflects each real customer's CURRENT risk profile (as of
# application_train), not a reconstruction of their risk at the actual
# historical moment of each specific previous-application decision this
# notebook predicts (real DAYS_DECISION can be years in the past). This is
# NOT leakage of this notebook's own TARGET -- Notebook 01's TARGET is real
# payment difficulty on the customer's CURRENT loan; this notebook's TARGET
# is whether a HISTORICAL previous application was Approved, a different
# real outcome column entirely -- but it IS a real, stated limitation: the
# feature is a present-day risk signal correlated with, not a time-accurate
# reconstruction of, the applicant's risk at each historical decision point.
# Disclosed here and throughout the reporting package below, never hidden.
#
# Soft dependency: if Notebook 01 has not been run yet, this feature is
# skipped and this notebook trains exactly as before -- never a hard failure.
# ---------------------------------------------------------------------------
UPSTREAM_MODEL_PATH = ARTIFACTS_DIR / "notebook_01_champion_model.joblib"
PD_INTEGRATION_AVAILABLE = UPSTREAM_MODEL_PATH.exists()
UPSTREAM_CHAMPION = None
pd_by_customer = None
N_MATCHED_TO_PD = 0

if PD_INTEGRATION_AVAILABLE:
    try:
        cust_df, up_check_numeric, up_check_categorical = engineer_credit_default_features(app, bureau)
        bundle = joblib.load(UPSTREAM_MODEL_PATH)
        up_model = bundle["model"]
        up_ord_enc = bundle["ordinal_encoder"]
        up_imputer = bundle["imputer"]
        up_feature_cols = bundle["feature_cols"]
        up_numeric = bundle["numeric_features"]
        up_categorical = bundle["categorical_features"]
        UPSTREAM_CHAMPION = bundle["champion_name"]
        if up_numeric != up_check_numeric or up_categorical != up_check_categorical:
            raise ValueError(
                "Feature set built here does not match Notebook 01's trained feature set "
                "(the shared feature module changed after Notebook 01 was trained)."
            )
        _pdf_up = cust_df.select(["SK_ID_CURR"] + up_feature_cols).to_pandas()
        for c in up_categorical:
            _pdf_up[c] = _pdf_up[c].astype(object).fillna("Missing").astype(str).astype("category")
        for c in up_numeric:
            _pdf_up[c] = _pdf_up[c].astype("float32")
        _X_up = _pdf_up[up_feature_cols].copy()
        if up_categorical:
            _X_up[up_categorical] = up_ord_enc.transform(_pdf_up[up_categorical].astype(str))
        _X_up[up_numeric] = up_imputer.transform(_pdf_up[up_numeric])
        _pd_scores = np.clip(up_model.predict_proba(_X_up)[:, 1], 1e-6, 1 - 1e-6)
        pd_by_customer = pl.DataFrame({
            "SK_ID_CURR": _pdf_up["SK_ID_CURR"].to_numpy(),
            "UPSTREAM_PD_FROM_NB01": _pd_scores,
        })
        print(f"[INTERDEPENDENCY] Real PD scored from Notebook 01's champion ({UPSTREAM_CHAMPION}) for "
              f"{pd_by_customer.height:,} real customers -- to be joined onto this notebook's real in-scope "
              f"rows by SK_ID_CURR as a new real input feature, UPSTREAM_PD_FROM_NB01. Temporal-snapshot "
              f"caveat: reflects each customer's CURRENT risk, not their risk reconstructed at each "
              f"historical decision's actual moment (see comment block above).")
        del cust_df
    except Exception as e:
        print(f"[INTERDEPENDENCY] Skipped: could not score with Notebook 01's model "
              f"({type(e).__name__}: {e}). This notebook trains without this feature, exactly as before.")
        PD_INTEGRATION_AVAILABLE = False
        pd_by_customer = None
else:
    print(f"[INTERDEPENDENCY] Skipped: Notebook 01 has not been run yet on this machine "
          f"({UPSTREAM_MODEL_PATH.name} not found). This is a soft dependency -- this notebook trains "
          f"standalone exactly as before. Run Notebook 01 first, then re-run this cell, to add the real "
          f"UPSTREAM_PD_FROM_NB01 feature.")

if PD_INTEGRATION_AVAILABLE and pd_by_customer is not None:
    df = df.join(pd_by_customer, on="SK_ID_CURR", how="left")
    N_MATCHED_TO_PD = int(df["UPSTREAM_PD_FROM_NB01"].is_not_null().sum())
    print(f"[INTERDEPENDENCY] {N_MATCHED_TO_PD:,} / {N_JOINED:,} real in-scope rows "
          f"({N_MATCHED_TO_PD / N_JOINED:.2%}) matched to a real current PD from Notebook 01 by SK_ID_CURR "
          f"(any unmatched row keeps UPSTREAM_PD_FROM_NB01 = null, handled by the same median imputer as "
          f"every other real numeric feature below -- never dropped or hard-failed).")

# ---------------------------------------------------------------------------
# SECTION 7 — Final feature selection & assembly (WARP: vectorized, no leakage)
# ---------------------------------------------------------------------------
NUMERIC_FEATURES = [c for c in [
    "AMT_ANNUITY", "AMT_APPLICATION", "AMT_CREDIT", "AMT_DOWN_PAYMENT", "AMT_GOODS_PRICE",
    "HOUR_APPR_PROCESS_START", "RATE_DOWN_PAYMENT", "SELLERPLACE_AREA", "CNT_PAYMENT",
    "DAYS_DECISION", "AMT_INCOME_TOTAL", "APPLICANT_AGE_YEARS", "EXT_SOURCE_2",
] if c in df.columns] + CREDIT_HISTORY_FEATURES
if PD_INTEGRATION_AVAILABLE and pd_by_customer is not None and "UPSTREAM_PD_FROM_NB01" in df.columns:
    NUMERIC_FEATURES = NUMERIC_FEATURES + ["UPSTREAM_PD_FROM_NB01"]
    print(f"[INTERDEPENDENCY] UPSTREAM_PD_FROM_NB01 added as a real input feature -- this notebook's model "
          f"now genuinely consumes Notebook 01's real, independently-trained default-risk signal.")
CATEGORICAL_FEATURES = [c for c in [
    "NAME_CONTRACT_TYPE", "WEEKDAY_APPR_PROCESS_START", "NAME_CASH_LOAN_PURPOSE",
    "NAME_PAYMENT_TYPE", "NAME_TYPE_SUITE", "NAME_CLIENT_TYPE", "NAME_GOODS_CATEGORY",
    "NAME_PORTFOLIO", "NAME_PRODUCT_TYPE", "CHANNEL_TYPE", "NAME_SELLER_INDUSTRY",
    "NAME_YIELD_GROUP", "PRODUCT_COMBINATION", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS",
    "OCCUPATION_TYPE",
] if c in df.columns]
assert not (set(NUMERIC_FEATURES + CATEGORICAL_FEATURES) & LEAKAGE_COLS), "Leakage column slipped into features"
FEATURE_COLS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

pdf = df.select(["SK_ID_PREV", "SK_ID_CURR", "TARGET"] + FEATURE_COLS).to_pandas()
for c in CATEGORICAL_FEATURES:
    pdf[c] = pdf[c].astype(object).fillna("Missing").astype(str).astype("category")
for c in NUMERIC_FEATURES:
    pdf[c] = pdf[c].astype("float32")

print(f"[FEATURES] {len(NUMERIC_FEATURES)} numeric + {len(CATEGORICAL_FEATURES)} categorical "
      f"= {len(FEATURE_COLS)} total (leakage columns explicitly excluded: {sorted(LEAKAGE_COLS)})")

# ---------------------------------------------------------------------------
# SECTION 8 — Train / holdout split (stratified, seeded)
# ---------------------------------------------------------------------------
X = pdf[FEATURE_COLS]
y = pdf["TARGET"].astype(int)
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=SEED
)
print(f"[SPLIT] train={X_train.shape[0]:,} holdout={X_holdout.shape[0]:,} "
      f"(holdout approval rate {y_holdout.mean():.4%})")

ord_enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
imputer = SimpleImputer(strategy="median")
X_train_enc, X_holdout_enc = X_train.copy(), X_holdout.copy()
if CATEGORICAL_FEATURES:
    X_train_enc[CATEGORICAL_FEATURES] = ord_enc.fit_transform(X_train[CATEGORICAL_FEATURES].astype(str))
    X_holdout_enc[CATEGORICAL_FEATURES] = ord_enc.transform(X_holdout[CATEGORICAL_FEATURES].astype(str))
X_train_enc[NUMERIC_FEATURES] = imputer.fit_transform(X_train[NUMERIC_FEATURES])
X_holdout_enc[NUMERIC_FEATURES] = imputer.transform(X_holdout[NUMERIC_FEATURES])

# ---------------------------------------------------------------------------
# SECTION 9 — Model candidate screening: TOP 4 real models only (reduced from
# a prior 6-model set). The 4 were chosen for real predictive strength on
# tabular credit data AND genuine multi-core parallelism: GradientBoosting
# Classifier (no n_jobs support at all in scikit-learn's implementation) and
# LogisticRegression (n_jobs has no effect on binary classification) are the
# two models this notebook dropped -- both would otherwise run pinned to a
# single thread while the rest of the configured CPU_CEILING_THREADS threads
# sit idle, a real, code-level cause of low observed CPU utilization during
# training, independent of the underlying hardware.
# ---------------------------------------------------------------------------
def make_candidate_models():
    return {
        "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=8, random_state=SEED,
                                                 n_jobs=CPU_CEILING_THREADS),
        "XGBoost": xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8,
                                      colsample_bytree=0.8, eval_metric="auc", random_state=SEED,
                                      n_jobs=CPU_CEILING_THREADS),
        "CatBoost": cb.CatBoostClassifier(iterations=300, depth=6, learning_rate=0.05, random_seed=SEED,
                                           verbose=False, thread_count=CPU_CEILING_THREADS,
                                           train_dir=str(ARTIFACTS_DIR / "_catboost_info"), allow_writing_files=False),
        "LightGBM": lgb.LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=SEED,
                                        n_jobs=CPU_CEILING_THREADS, verbose=-1),
    }

CANDIDATE_NAMES = list(make_candidate_models().keys())
print(f"[MODELS] Top-{len(CANDIDATE_NAMES)} real candidate set (every one genuinely multi-threaded, "
      f"each using the full {CPU_CEILING_THREADS}-thread WARP ceiling): {CANDIDATE_NAMES}")

# Stage A — fast, single real train/validation split across all 4 candidates
# (one fit each, not a full 5-fold commitment) to rank them cheaply before
# deciding which 2 are worth the full CV cost.
X_screen_train, X_screen_val, y_screen_train, y_screen_val = train_test_split(
    X_train_enc, y_train, test_size=0.20, stratify=y_train, random_state=SEED
)
screening_results = {}
for name, model in make_candidate_models().items():
    t_screen = time.time()
    model.fit(X_screen_train, y_screen_train)
    proba = model.predict_proba(X_screen_val)[:, 1]
    auc = float(roc_auc_score(y_screen_val, proba))
    screening_results[name] = {"screen_auc": auc, "fit_seconds": round(time.time() - t_screen, 2)}
    print(f"[SCREEN] {name:<14} single-split real AUC = {auc:.4f} (fit in "
          f"{screening_results[name]['fit_seconds']}s using {CPU_CEILING_THREADS} threads)")

TOP2_NAMES = sorted(screening_results, key=lambda n: screening_results[n]["screen_auc"], reverse=True)[:2]
print(f"[SCREEN] Top 2 of {len(CANDIDATE_NAMES)} advancing to real 5-fold CV: {TOP2_NAMES}")

# ---------------------------------------------------------------------------
# SECTION 9B — Champion selection: real 5-fold StratifiedKFold CV computed
# ONLY for the top-2 screened candidates (not all 4) -- roughly halves the CV
# cost of the prior 6-model x 5-fold benchmark (30 fits) down to 2-model x
# 5-fold (10 fits), directly speeding up a real run on top of the hardware fix
# above, while still giving the champion a real, honest 5-fold CV result (not
# just the single-split screen).
# ---------------------------------------------------------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

cv_results = {}
for name in TOP2_NAMES:
    fold_aucs = []
    for tr_idx, va_idx in cv.split(X_train_enc, y_train):
        model = make_candidate_models()[name]
        model.fit(X_train_enc.iloc[tr_idx], y_train.iloc[tr_idx])
        proba = model.predict_proba(X_train_enc.iloc[va_idx])[:, 1]
        fold_aucs.append(roc_auc_score(y_train.iloc[va_idx], proba))
    cv_results[name] = {"mean_auc": float(np.mean(fold_aucs)), "std_auc": float(np.std(fold_aucs)),
                         "fold_aucs": [float(a) for a in fold_aucs]}
    print(f"[CV] {name:<14} mean AUC = {cv_results[name]['mean_auc']:.4f} (+/- {cv_results[name]['std_auc']:.4f}) "
          f"across 5 real folds: {[round(a, 4) for a in fold_aucs]}")

CHAMPION_NAME = max(cv_results, key=lambda k: cv_results[k]["mean_auc"])
RUNNER_UP_NAME = [n for n in TOP2_NAMES if n != CHAMPION_NAME][0]
print(f"[CHAMPION] {CHAMPION_NAME} selected by highest mean 5-fold CV AUC among the top-2 "
      f"screened candidates (runner-up: {RUNNER_UP_NAME})")

# ---------------------------------------------------------------------------
# SECTION 10 — Retrain champion on full train, evaluate on true holdout
# ---------------------------------------------------------------------------
champion = make_candidate_models()[CHAMPION_NAME]
champion.fit(X_train_enc, y_train)
holdout_proba = champion.predict_proba(X_holdout_enc)[:, 1]
holdout_pred = (holdout_proba >= 0.5).astype(int)
y_holdout_arr = y_holdout.to_numpy()
metrics = {
    "roc_auc": float(roc_auc_score(y_holdout, holdout_proba)),
    "precision": float(precision_score(y_holdout, holdout_pred, zero_division=0)),
    "recall": float(recall_score(y_holdout, holdout_pred, zero_division=0)),
    "f1": float(f1_score(y_holdout, holdout_pred, zero_division=0)),
    "brier_score": float(brier_score_loss(y_holdout, holdout_proba)),
}
print(f"[HOLDOUT] {CHAMPION_NAME} real holdout metrics: {json.dumps(metrics, indent=2)}")

# ---------------------------------------------------------------------------
# SECTION 11 — Inline charts (vivid multicolor, per the standing chart-style rule)
# ---------------------------------------------------------------------------
fpr, tpr, _ = roc_curve(y_holdout, holdout_proba)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(fpr, tpr, label=f"{CHAMPION_NAME} (AUC={metrics['roc_auc']:.4f})",
             color=VIVID_PALETTE[0], linewidth=2.5)
axes[0].plot([0, 1], [0, 1], "--", color="#898781", alpha=0.7)
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("Approval-Model Holdout ROC (real data)"); axes[0].legend()

names = list(cv_results.keys())
means = [cv_results[n]["mean_auc"] for n in names]
stds = [cv_results[n]["std_auc"] for n in names]
bar_colors = [VIVID_PALETTE[1] if n == CHAMPION_NAME else c for n, c in zip(names, _palette(len(names)))]
axes[1].barh(names, means, xerr=stds, color=bar_colors)
axes[1].set_xlabel("Mean CV ROC-AUC"); axes[1].set_title("Top-2 5-Fold CV Champion Selection")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "notebook_02_benchmark_chart.png", dpi=110)
plt.show()

# --- Stage-A screening chart: all 4 real candidates, top 2 highlighted ------
fig, ax = plt.subplots(figsize=(8.5, 5))
_screen_names = list(screening_results.keys())
_screen_aucs = [screening_results[n]["screen_auc"] for n in _screen_names]
_screen_colors = [VIVID_PALETTE[1] if n in TOP2_NAMES else "#a9a9a9" for n in _screen_names]
ax.bar(_screen_names, _screen_aucs, color=_screen_colors)
ax.set_ylabel("Single-split real AUC")
ax.set_title(f"Stage A: Top-{len(CANDIDATE_NAMES)} Model Screening (top 2 advance to real 5-fold CV)")
plt.tight_layout()
screening_chart_path = REPORTS_DIR / "notebook_02_screening_chart.png"
plt.savefig(screening_chart_path, dpi=110)
plt.show()

# --- CV Report chart: real per-fold ROC-AUC for the top-2 models, grouped by
# fold -- shows real fold-to-fold variance, not just the mean/std above. -----
fig, ax = plt.subplots(figsize=(9, 5.5))
_fold_x = np.arange(5)
_bar_w = 0.35
for i, name in enumerate(TOP2_NAMES):
    offset = (i - 0.5) * _bar_w
    color = VIVID_PALETTE[1] if name == CHAMPION_NAME else VIVID_PALETTE[4]
    ax.bar(_fold_x + offset, cv_results[name]["fold_aucs"], width=_bar_w, label=name, color=color)
ax.set_xticks(_fold_x); ax.set_xticklabels([f"Fold {i}" for i in range(1, 6)])
ax.set_ylabel("Real ROC-AUC")
ax.set_title(f"5-Fold CV Report — Top 2 Models (champion: {CHAMPION_NAME})")
ax.legend()
plt.tight_layout()
cv_report_chart_path = REPORTS_DIR / "notebook_02_cv_report.png"
plt.savefig(cv_report_chart_path, dpi=110)
plt.show()

cv_report_rows = [
    {"model": name, "fold": fold_i, "roc_auc": auc}
    for name in TOP2_NAMES
    for fold_i, auc in enumerate(cv_results[name]["fold_aucs"], start=1)
]
cv_report_df = pd.DataFrame(cv_report_rows)
print(f"[CV-REPORT] Real per-fold CV report built for {TOP2_NAMES} ({len(cv_report_df)} rows).")

# ---------------------------------------------------------------------------
# SECTION 11B — SHAP Explainability (champion model only, per the standing
# "top 4 models + champion-only SHAP/LIME" specification). shap.TreeExplainer
# supports every one of this notebook's 4 candidate model types natively.
# ---------------------------------------------------------------------------
SHAP_SAMPLE_N = min(300, len(X_holdout_enc))
X_shap_sample = X_holdout_enc.sample(n=SHAP_SAMPLE_N, random_state=SEED)
shap_explainer = shap.TreeExplainer(champion)
_raw_shap = shap_explainer.shap_values(X_shap_sample)
if isinstance(_raw_shap, list):
    SHAP_VALUES = np.asarray(_raw_shap[1])          # positive ("Approved") class
elif isinstance(_raw_shap, np.ndarray) and _raw_shap.ndim == 3:
    SHAP_VALUES = _raw_shap[:, :, 1]                 # (rows, features, classes) -> positive class
else:
    SHAP_VALUES = np.asarray(_raw_shap)

mean_abs_shap = np.abs(SHAP_VALUES).mean(axis=0)
shap_importance_df = pd.DataFrame(
    {"feature": FEATURE_COLS, "mean_abs_shap": mean_abs_shap}
).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
TOP_SHAP_FEATURES = shap_importance_df.head(15)

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(TOP_SHAP_FEATURES["feature"][::-1], TOP_SHAP_FEATURES["mean_abs_shap"][::-1],
        color=_palette(len(TOP_SHAP_FEATURES)))
ax.set_xlabel("Mean |SHAP value| (real impact on predicted approval probability)")
ax.set_title(f"SHAP Feature Importance — Champion ({CHAMPION_NAME}), "
             f"real sample of {SHAP_SAMPLE_N} holdout applications")
plt.tight_layout()
shap_summary_path = REPORTS_DIR / "notebook_02_shap_summary.png"
plt.savefig(shap_summary_path, dpi=110)
plt.show()

shap_beeswarm_path = None
try:
    plt.figure(figsize=(9, 7))
    shap.summary_plot(SHAP_VALUES, X_shap_sample, feature_names=FEATURE_COLS, show=False, max_display=15)
    plt.tight_layout()
    shap_beeswarm_path = REPORTS_DIR / "notebook_02_shap_beeswarm.png"
    plt.savefig(shap_beeswarm_path, dpi=110)
    plt.close()
except Exception as e:
    print(f"[SHAP] Beeswarm detail plot skipped ({type(e).__name__}: {e}); "
          f"the bar-chart summary above is unaffected — explainability chart only, no correctness impact.")
print(f"[SHAP] Real SHAP explainability computed for champion {CHAMPION_NAME} on a real "
      f"{SHAP_SAMPLE_N}-row holdout sample. Top real driver: {TOP_SHAP_FEATURES.iloc[0]['feature']} "
      f"(mean |SHAP|={TOP_SHAP_FEATURES.iloc[0]['mean_abs_shap']:.4f}).")

# ---------------------------------------------------------------------------
# SECTION 11C — LIME Explainability (champion model only). Local, instance-
# level explanations on real representative holdout applications selected
# from this run's own real predictions -- never fabricated.
# ---------------------------------------------------------------------------
CATEGORICAL_FEATURE_IDX = [FEATURE_COLS.index(c) for c in CATEGORICAL_FEATURES]
lime_explainer = LimeTabularExplainer(
    training_data=X_train_enc.to_numpy(),
    feature_names=FEATURE_COLS,
    categorical_features=CATEGORICAL_FEATURE_IDX,
    class_names=["Not Approved", "Approved"],
    mode="classification",
    random_state=SEED,
)

_lime_pool = pd.DataFrame({
    "idx": np.arange(len(X_holdout_enc)), "actual": y_holdout_arr,
    "proba": holdout_proba, "pred": holdout_pred,
})
_lime_cases = {}
_correct_approve = _lime_pool[(_lime_pool.actual == 1) & (_lime_pool.pred == 1)]
if len(_correct_approve):
    _lime_cases["Most-confident correct approval"] = int(_correct_approve.loc[_correct_approve.proba.idxmax(), "idx"])
_correct_decline = _lime_pool[(_lime_pool.actual == 0) & (_lime_pool.pred == 0)]
if len(_correct_decline):
    _lime_cases["Most-confident correct decline"] = int(_correct_decline.loc[_correct_decline.proba.idxmin(), "idx"])
_misclassified = _lime_pool[_lime_pool.actual != _lime_pool.pred]
if len(_misclassified):
    _lime_cases["A real misclassified application"] = int(_misclassified.sample(n=1, random_state=SEED)["idx"].iloc[0])

LIME_EXPLANATIONS = []
_lime_fig_paths = []
for case_label, row_idx in _lime_cases.items():
    instance = X_holdout_enc.iloc[row_idx].to_numpy()
    exp = lime_explainer.explain_instance(instance, champion.predict_proba, num_features=8)
    for feat, weight in exp.as_list():
        LIME_EXPLANATIONS.append({"case": case_label, "feature_condition": feat, "weight": float(weight)})
    try:
        fig_l = exp.as_pyplot_figure()
        fig_l.suptitle(f"LIME — {case_label} (real holdout row)")
        fig_l.tight_layout()
        p = REPORTS_DIR / f"notebook_02_lime_{len(_lime_fig_paths)}.png"
        fig_l.savefig(p, dpi=110)
        plt.close(fig_l)
        _lime_fig_paths.append(p)
    except Exception as e:
        print(f"[LIME] Plot skipped for '{case_label}' ({type(e).__name__}: {e}); table rows above are unaffected.")

lime_explanations_df = pd.DataFrame(LIME_EXPLANATIONS)
lime_panel_path = _lime_fig_paths[0] if _lime_fig_paths else None
print(f"[LIME] Real local explanations computed for champion {CHAMPION_NAME} on "
      f"{len(_lime_cases)} representative real holdout case(s): {list(_lime_cases.keys())}")

# ---------------------------------------------------------------------------
# SECTION 12 — Statistical Validation & Deployment Readiness (SOP Stage 4)
# Bootstrap AUC confidence interval, calibration-by-decile, split-half PSI —
# every figure computed live from this run's real holdout predictions.
# ---------------------------------------------------------------------------
rng = np.random.default_rng(SEED)
N_BOOTSTRAP = 1000
boot_aucs = []
for _ in range(N_BOOTSTRAP):
    idx = rng.integers(0, len(y_holdout_arr), len(y_holdout_arr))
    y_bs = y_holdout_arr[idx]
    if len(np.unique(y_bs)) < 2:
        continue  # degenerate resample (single class) — can occur on tiny fixtures, skip
    boot_aucs.append(roc_auc_score(y_bs, holdout_proba[idx]))
boot_aucs = np.array(boot_aucs)
AUC_CI_LOW, AUC_CI_HIGH = (float(np.percentile(boot_aucs, 2.5)), float(np.percentile(boot_aucs, 97.5))) \
    if len(boot_aucs) > 0 else (float("nan"), float("nan"))
print(f"[VALIDATION] Real {len(boot_aucs)}-resample bootstrap 95% CI on holdout ROC-AUC: "
      f"[{AUC_CI_LOW:.4f}, {AUC_CI_HIGH:.4f}]")

calib_df = pd.DataFrame({"y": y_holdout_arr, "p": holdout_proba})
calib_df["decile"] = pd.qcut(calib_df["p"], q=10, labels=False, duplicates="drop")
calib_summary = (
    calib_df.groupby("decile")
    .agg(mean_predicted=("p", "mean"), actual_rate=("y", "mean"), n=("y", "size"))
    .reset_index()
)
MEAN_CALIBRATION_GAP = float((calib_summary["mean_predicted"] - calib_summary["actual_rate"]).abs().mean())
print(f"[VALIDATION] Real mean calibration gap (|predicted - actual| across "
      f"{len(calib_summary)} deciles): {MEAN_CALIBRATION_GAP:.4f}")

half_idx = rng.permutation(len(holdout_proba))
half_a = holdout_proba[half_idx[: len(half_idx) // 2]]
half_b = holdout_proba[half_idx[len(half_idx) // 2:]]
_bins = np.linspace(0, 1, 11)
def _psi(a, b, bins):
    a_counts, _ = np.histogram(a, bins=bins)
    b_counts, _ = np.histogram(b, bins=bins)
    a_pct = np.clip(a_counts / max(a_counts.sum(), 1), 1e-4, None)
    b_pct = np.clip(b_counts / max(b_counts.sum(), 1), 1e-4, None)
    return float(np.sum((a_pct - b_pct) * np.log(a_pct / b_pct)))
SPLIT_HALF_PSI = _psi(half_a, half_b, _bins)
print(f"[VALIDATION] Real split-half PSI on holdout score distribution: {SPLIT_HALF_PSI:.4f}")

# ASSUMPTION thresholds (industry convention, not fabricated data — same explicit
# labeling standard as every other assumption in this suite):
CALIBRATION_GAP_THRESHOLD = 0.10   # ASSUMPTION — informal convention: <0.10 mean gap considered acceptable
PSI_STABILITY_THRESHOLD = 0.10     # ASSUMPTION — standard PSI convention: <0.10 stable, 0.10-0.25 moderate shift, >0.25 significant shift
deployment_checks = [
    ("auc_ci_lower_above_random", AUC_CI_LOW > 0.5),
    ("mean_calibration_gap_acceptable", MEAN_CALIBRATION_GAP < CALIBRATION_GAP_THRESHOLD),
    ("split_half_psi_stable", SPLIT_HALF_PSI < PSI_STABILITY_THRESHOLD),
]
DEPLOYMENT_READY = all(ok for _, ok in deployment_checks)
_failed_deployment_checks = [name for name, ok in deployment_checks if not ok]
# NOTE: see the identical comment in pipeline_body.py (Notebook 01) -- this
# "deployment_checks" family is a separate, stricter STATISTICAL ROBUSTNESS
# gate from the "integrity_checks" structural pipeline-sanity family reported
# later in this notebook. Fixed for consistency during the hardening pass
# (see CHANGELOG.md) even though this notebook's checks currently pass, so
# the verdict text stays unambiguous if a future real run fails one.
DEPLOYMENT_VERDICT = (
    "RECOMMENDED FOR PRODUCTION" if DEPLOYMENT_READY
    else "NOT RECOMMENDED FOR PRODUCTION YET — failed: " + ", ".join(_failed_deployment_checks) +
         " (this is a separate, stricter statistical-robustness gate, distinct from the "
         "structural pipeline integrity checks reported elsewhere in this notebook's output; "
         "failing here does not indicate a code defect, and passing all integrity checks does "
         "not imply this gate passed -- expected and informational on small or noisy "
         "real/synthetic samples, see this problem's MODEL_CARD.md)"
)
for name, ok in deployment_checks:
    print(f"[VALIDATION-CHECK] {name}: {'PASS' if ok else 'FAIL'}")
print(f"[VALIDATION] Deployment readiness verdict: {DEPLOYMENT_VERDICT}")

# ---------------------------------------------------------------------------
# SECTION 13 — Integrity self-checks (fail loudly, never silently pass bad state)
# Run BEFORE Stage 5 reporting so the real check results can be embedded in the
# Word/Excel/HTML package rather than a placeholder.
# ---------------------------------------------------------------------------
checks = [
    ("target_is_binary", set(y.unique().tolist()) <= {0, 1}),
    ("no_leakage_columns", not (set(FEATURE_COLS) & LEAKAGE_COLS)),
    ("holdout_size_matches_split", X_holdout.shape[0] == y_holdout.shape[0]),
    ("champion_auc_above_random", metrics["roc_auc"] > 0.5),
    ("cv_fold_count_correct", all(len(v["fold_aucs"]) == 5 for v in cv_results.values())),
    ("no_null_features_after_impute", not np.isnan(X_train_enc[NUMERIC_FEATURES].to_numpy()).any()),
    ("join_scope_reported", 0.0 < join_rate <= 1.0),
    ("credit_history_features_integrated", len(CREDIT_HISTORY_FEATURES) > 0),
    ("servicing_features_leave_one_out_nonnegative",
     bool((df["POS_CNT_RECORDS"] >= 0).all() and (df["INSTAL_CNT_PAYMENTS"] >= 0).all()
          and (df["CC_CNT_RECORDS"] >= 0).all())),
    ("candidate_screen_is_top4", len(CANDIDATE_NAMES) == 4),
    ("cv_computed_for_top2_only", len(cv_results) == 2),
    ("champion_in_screened_top2", CHAMPION_NAME in TOP2_NAMES),
    ("shap_values_finite", bool(np.isfinite(SHAP_VALUES).all())),
    ("shap_feature_count_matches", len(shap_importance_df) == len(FEATURE_COLS)),
    ("lime_explanations_computed", len(LIME_EXPLANATIONS) > 0),
    ("cpu_thread_ceiling_applied_before_import",
     os.environ.get("OMP_NUM_THREADS") == str(CPU_CEILING_THREADS)),
]
if PD_INTEGRATION_AVAILABLE and pd_by_customer is not None and "UPSTREAM_PD_FROM_NB01" in NUMERIC_FEATURES:
    checks.extend([
        ("upstream_pd_feature_integrated", "UPSTREAM_PD_FROM_NB01" in FEATURE_COLS),
        ("upstream_pd_matched_rows_positive", N_MATCHED_TO_PD > 0),
    ])
for name, ok in checks:
    print(f"[CHECK] {name}: {'PASS' if ok else 'FAIL'}")
failed = [n for n, ok in checks if not ok]
if failed:
    raise AssertionError(f"Integrity checks failed: {failed}")

# ---------------------------------------------------------------------------
# SECTION 14 — Financial-Impact Reporting & Packaging (SOP Stage 5)
# Real CSV outputs + a Word report + an Excel workbook (formula-driven) + an
# HTML dashboard, generated from this run's own real computed results via the
# shared src/reporting module (HYPER: built once, reused by every notebook).
# ---------------------------------------------------------------------------
tn, fp_n, fn_n, tp_n = confusion_matrix(y_holdout_arr, holdout_pred).ravel()

if "AMT_CREDIT" in X_holdout.columns:
    holdout_amt_credit = X_holdout["AMT_CREDIT"].to_numpy()
    tp_mask = (y_holdout_arr == 1) & (holdout_pred == 1)
    fn_mask = (y_holdout_arr == 1) & (holdout_pred == 0)
    fp_mask = (y_holdout_arr == 0) & (holdout_pred == 1)
    TOTAL_HOLDOUT_VOLUME = float(np.nansum(holdout_amt_credit))
    CORRECTLY_APPROVED_VOLUME = float(np.nansum(holdout_amt_credit[tp_mask]))
    MISSED_APPROVAL_VOLUME = float(np.nansum(holdout_amt_credit[fn_mask]))
    WRONGLY_APPROVED_VOLUME = float(np.nansum(holdout_amt_credit[fp_mask]))
else:
    holdout_amt_credit = np.zeros(len(y_holdout_arr))
    TOTAL_HOLDOUT_VOLUME = CORRECTLY_APPROVED_VOLUME = MISSED_APPROVAL_VOLUME = WRONGLY_APPROVED_VOLUME = 0.0

CAPTURE_RATE = float(tp_n / (tp_n + fn_n)) if (tp_n + fn_n) > 0 else 0.0
print(f"[IMPACT] Real holdout credit volume: total=${TOTAL_HOLDOUT_VOLUME:,.0f} "
      f"correctly-approved(TP)=${CORRECTLY_APPROVED_VOLUME:,.0f} missed-approval(FN)=${MISSED_APPROVAL_VOLUME:,.0f} "
      f"wrongly-flagged-approved(FP)=${WRONGLY_APPROVED_VOLUME:,.0f} | capture rate={CAPTURE_RATE:.2%}")

ASSUMPTIONS = {
    "CLASSIFICATION_THRESHOLD": 0.5,
    "AVG_PROCESSING_COST_PER_MANUAL_REVIEW": 25.0,
}
ASSUMPTION_NOTES = {
    "CLASSIFICATION_THRESHOLD": "Standard 0.5 decision boundary — same threshold used for precision/recall above",
    "AVG_PROCESSING_COST_PER_MANUAL_REVIEW": "Illustrative operations-cost convention for a manual underwriting "
                                              "review — applied only to the false-alarm (FP) count, never blended "
                                              "into the real volume figures above",
}
ESTIMATED_MANUAL_REVIEW_COST_AVOIDED = float(tp_n) * ASSUMPTIONS["AVG_PROCESSING_COST_PER_MANUAL_REVIEW"]

holdout_sk_prev = pdf.loc[X_holdout.index, "SK_ID_PREV"].to_numpy()
holdout_predictions_df = pd.DataFrame({
    "SK_ID_PREV": holdout_sk_prev,
    "actual_target": y_holdout_arr,
    "predicted_probability": holdout_proba,
    "predicted_label": holdout_pred,
    "amt_credit": holdout_amt_credit,
})
holdout_predictions_df["outcome"] = np.select(
    [
        (holdout_predictions_df["actual_target"] == 1) & (holdout_predictions_df["predicted_label"] == 1),
        (holdout_predictions_df["actual_target"] == 1) & (holdout_predictions_df["predicted_label"] == 0),
        (holdout_predictions_df["actual_target"] == 0) & (holdout_predictions_df["predicted_label"] == 1),
    ],
    ["Correctly Approved (TP)", "Missed Approval (FN)", "Wrongly Flagged Approve (FP)"],
    default="Correctly Declined (TN)",
)
model_comparison_df = pd.DataFrame([
    {"model": name, "mean_cv_auc": cv_results[name]["mean_auc"], "std_cv_auc": cv_results[name]["std_auc"]}
    for name in cv_results
])
screening_df = pd.DataFrame([
    {"model": name, "screen_auc": v["screen_auc"], "fit_seconds": v["fit_seconds"],
     "advanced_to_cv": name in TOP2_NAMES}
    for name, v in screening_results.items()
])
calibration_df = calib_summary.rename(columns={"decile": "score_decile"})

# --- Real narrative "stories" (3-4 sentences each, built only from this run's
# own computed numbers via f-strings -- never invented commentary) + real
# SMART-format recommendations (HYPER: computed once here, rendered three ways
# across Word / Excel / HTML). ------------------------------------------------
_champion_gap = cv_results[CHAMPION_NAME]["mean_auc"] - cv_results[RUNNER_UP_NAME]["mean_auc"]
_weakest_screened = min(screening_results, key=lambda n: screening_results[n]["screen_auc"])
STORY_MODEL_CHART = [
    f"{CHAMPION_NAME} wins real 5-fold CV among the top-2 screened candidates with a mean ROC-AUC of "
    f"{cv_results[CHAMPION_NAME]['mean_auc']:.4f} (+/- {cv_results[CHAMPION_NAME]['std_auc']:.4f}), "
    f"{_champion_gap:.4f} ahead of the runner-up, {RUNNER_UP_NAME}.",
    f"Of the {len(CANDIDATE_NAMES)} real candidates screened in Stage A, {_weakest_screened} scored weakest "
    f"({screening_results[_weakest_screened]['screen_auc']:.4f} single-split AUC) and did not advance to the "
    f"full 5-fold CV — this two-stage screen is what keeps this notebook to a real top-4/top-2 benchmark "
    f"instead of committing every candidate to the full CV cost.",
    f"On the true holdout set, {CHAMPION_NAME} scores {metrics['roc_auc']:.4f} ROC-AUC "
    f"(95% bootstrap CI [{AUC_CI_LOW:.4f}, {AUC_CI_HIGH:.4f}]), consistent with its CV performance.",
    f"Deployment readiness verdict: {DEPLOYMENT_VERDICT}.",
]
STORY_SHAP_CHART = [
    f"Real SHAP explainability was computed for the champion model ({CHAMPION_NAME}) only, on a real "
    f"{SHAP_SAMPLE_N}-row sample of the holdout set — per the standing rule that champion-only explainability "
    f"is sufficient once model selection is already narrowed to a top-2 CV comparison.",
    f"The single strongest real driver of predicted approval probability is "
    f"{TOP_SHAP_FEATURES.iloc[0]['feature']} (mean |SHAP| = {TOP_SHAP_FEATURES.iloc[0]['mean_abs_shap']:.4f}), "
    f"followed by {TOP_SHAP_FEATURES.iloc[1]['feature']} "
    f"(mean |SHAP| = {TOP_SHAP_FEATURES.iloc[1]['mean_abs_shap']:.4f}).",
    "Positive SHAP values push a real application's predicted probability toward approval; negative values "
    "push it toward decline — see the beeswarm detail chart for the real direction of each feature's effect.",
]
STORY_LIME_CHART = [
    f"Real local (instance-level) LIME explanations were computed for {len(_lime_cases)} representative real "
    f"holdout applications selected from this run's own predictions: {', '.join(_lime_cases.keys())}.",
    "Unlike SHAP's global feature-importance ranking above, LIME shows exactly which real feature values "
    "moved THIS SPECIFIC application's prediction — useful for explaining an individual underwriting decision "
    "to an applicant or auditor.",
    "Full per-feature weights for every case are in the LIME Instance Explanations table/sheet below.",
]
STORY_MISSING_CHART = [
    f"{len(null_counts)} of {df.width} real in-scope columns have at least one missing value; the worst, "
    f"{top_missing.iloc[0]['column']}, is missing in {top_missing.iloc[0]['pct_null']:.1%} of in-scope rows.",
    f"The top 5 columns by missingness average {top_missing.head(5)['pct_null'].mean():.1%} missing.",
    f"{len(CREDIT_HISTORY_FEATURES)} new applicant credit-history features were integrated from 5 additional "
    f"real Home Credit tables (bureau, bureau_balance, POS_CASH_balance, installments_payments, "
    f"credit_card_balance) to close this gap and add real signal.",
    "Use the filter above to switch between the top-15 and top-5 views of this same real ranking.",
]
STORY_CALIB_CHART = [
    f"Mean calibration gap across {len(calib_summary)} real score deciles is {MEAN_CALIBRATION_GAP:.4f} "
    f"(ASSUMPTION threshold for 'acceptable': <{CALIBRATION_GAP_THRESHOLD}), so the model is "
    f"{'well-calibrated' if MEAN_CALIBRATION_GAP < CALIBRATION_GAP_THRESHOLD else 'not yet well-calibrated'}.",
    f"Split-half PSI on the holdout score distribution is {SPLIT_HALF_PSI:.4f} "
    f"(ASSUMPTION threshold: <{PSI_STABILITY_THRESHOLD}), indicating "
    f"{'a stable' if SPLIT_HALF_PSI < PSI_STABILITY_THRESHOLD else 'a shifting'} score distribution.",
    f"The top decile shows a real mean predicted approval probability of "
    f"{calib_summary['mean_predicted'].iloc[-1]:.2%} against an actual observed rate of "
    f"{calib_summary['actual_rate'].iloc[-1]:.2%}.",
]
STORY_VOLUME_CHART = [
    f"Of ${TOTAL_HOLDOUT_VOLUME:,.0f} in real total holdout requested credit volume, the model correctly "
    f"identifies ${CORRECTLY_APPROVED_VOLUME:,.0f} of volume that was really approved "
    f"(capture rate {CAPTURE_RATE:.1%} of real approvals).",
    f"${MISSED_APPROVAL_VOLUME:,.0f} of volume is missed (real approvals the model would have declined) and "
    f"${WRONGLY_APPROVED_VOLUME:,.0f} is flagged as approvable on applications that were really declined.",
    f"ASSUMPTION (illustrative manual-review cost, ${ASSUMPTIONS['AVG_PROCESSING_COST_PER_MANUAL_REVIEW']:.0f}/review): "
    f"applied only to correctly-identified approvals, illustrative manual-review cost avoided = "
    f"${ESTIMATED_MANUAL_REVIEW_COST_AVOIDED:,.0f}.",
    "Switch the view above to see the same three outcomes by real application count instead of dollar volume.",
]
STORY_TARGET_CHART = [
    f"The real approval rate in scope is {APPROVAL_RATE:.2%}.",
    "This is the base rate ROC-AUC is benchmarked against, and why stratified sampling is used for both "
    "the CV folds and the train/holdout split.",
]
if PD_INTEGRATION_AVAILABLE and pd_by_customer is not None and "UPSTREAM_PD_FROM_NB01" in NUMERIC_FEATURES:
    _shap_rank = shap_importance_df.index[shap_importance_df["feature"] == "UPSTREAM_PD_FROM_NB01"].tolist()
    _shap_rank_text = (f"ranked #{_shap_rank[0] + 1} of {len(FEATURE_COLS)} real features by mean |SHAP|"
                        if _shap_rank else "present in the real feature set")
    STORY_INTERDEPENDENCY_CHART = [
        f"This notebook's model genuinely consumes Notebook 01's real, independently-trained default-risk "
        f"model ({UPSTREAM_CHAMPION}) as a new real input feature, UPSTREAM_PD_FROM_NB01 -- "
        f"{N_MATCHED_TO_PD:,} of {N_JOINED:,} real in-scope rows ({N_MATCHED_TO_PD / N_JOINED:.2%}) matched "
        f"to a real customer PD by SK_ID_CURR.",
        f"TEMPORAL-SNAPSHOT CAVEAT: this PD reflects each real customer's CURRENT risk profile, not a "
        f"reconstruction of their risk at the actual historical moment of each specific previous-application "
        f"decision being predicted -- a real, stated limitation, not leakage of this notebook's own TARGET "
        f"(a different real outcome column from Notebook 01's).",
        f"In this run's real SHAP explainability, UPSTREAM_PD_FROM_NB01 is {_shap_rank_text} -- see the SHAP "
        f"chart above for its real measured contribution to this model's predictions.",
    ]
else:
    STORY_INTERDEPENDENCY_CHART = [
        "Notebook 01 has not been run yet on this machine, so the real UPSTREAM_PD_FROM_NB01 cross-problem "
        "feature was skipped this run -- this is a soft dependency, and this notebook trained standalone, "
        "exactly as before, on its own real features. Run Notebook 01 first, then re-run this cell, to add "
        "the real feature and its SHAP contribution.",
    ]

DEPLOY_STATUS_WORD = "meets" if DEPLOYMENT_READY else "does not yet meet"
INSIGHTS = [
    {
        "headline": f"{CHAMPION_NAME} {DEPLOY_STATUS_WORD} the deployment-readiness bar",
        "specific": f"Holdout ROC-AUC {metrics['roc_auc']:.4f} with a 95% bootstrap CI of "
                    f"[{AUC_CI_LOW:.4f}, {AUC_CI_HIGH:.4f}]; deployment checks "
                    f"{sum(1 for _, ok in deployment_checks if ok)}/{len(deployment_checks)} PASS.",
        "measurable": f"Calibration gap {MEAN_CALIBRATION_GAP:.4f} vs. <{CALIBRATION_GAP_THRESHOLD} threshold; "
                      f"split-half PSI {SPLIT_HALF_PSI:.4f} vs. <{PSI_STABILITY_THRESHOLD} threshold.",
        "achievable": "No further tuning required this cycle." if DEPLOYMENT_READY else
                      "Investigate the failing check(s) above before promoting to production; "
                      "re-run this notebook after any fix to confirm.",
        "relevant": "Directly supports the underwriting-automation goal of Mega Project 1.",
        "timebound": "Verdict computed fresh on every run — re-check before each deployment cycle.",
    },
    {
        "headline": f"{len(CREDIT_HISTORY_FEATURES)} new credit-history features materially deepen this model",
        "specific": f"Integrated {len(BUREAU_FEATURES)} external-bureau features and {len(SERVICING_FEATURES)} "
                    f"leave-one-out-safe servicing features from 5 additional real Home Credit tables — total "
                    f"feature count is now {len(FEATURE_COLS)} (vs. the {len(FEATURE_COLS) - len(CREDIT_HISTORY_FEATURES)} "
                    f"used in the prior version of this notebook).",
        "measurable": f"Champion mean CV ROC-AUC: {cv_results[CHAMPION_NAME]['mean_auc']:.4f}.",
        "achievable": "Track this metric on the next full-data run to confirm the real accuracy lift.",
        "relevant": "Directly answers the standing instruction to integrate the relevant supplementary "
                    "datasets for higher real model accuracy.",
        "timebound": "Confirm on next full real-data run.",
    },
    {
        "headline": f"Close the data-quality gap on {top_missing.iloc[0]['column']}",
        "specific": f"{top_missing.iloc[0]['column']} is missing in {top_missing.iloc[0]['pct_null']:.1%} of "
                    f"{N_DF:,} real in-scope rows, the single worst column in this run.",
        "measurable": f"Track missingness on this column at each future run; "
                      f"{len(null_counts)} columns currently have at least one missing value.",
        "achievable": "Add or enforce a required capture field at intake, or source it from another real table.",
        "relevant": f"It is one of the correlation-with-TARGET columns tracked in this notebook's EDA "
                    f"(top-3 |r|: {target_corr.abs().sort_values(ascending=False).head(3).round(4).to_dict()}).",
        "timebound": "Target: before the next underwriting policy review cycle.",
    },
    {
        "headline": "Monitor wrongly-flagged-approve volume against manual review capacity",
        "specific": f"${WRONGLY_APPROVED_VOLUME:,.0f} of real holdout volume is on applications the model "
                    f"flags as approvable but that were really declined (false positives).",
        "measurable": f"Wrongly-flagged volume is {WRONGLY_APPROVED_VOLUME / max(CORRECTLY_APPROVED_VOLUME, 1):.2f}x "
                      f"the real correctly-approved volume — track this ratio on every run.",
        "achievable": "Route borderline-probability decisions to manual review rather than auto-approve, "
                      "using the 0.5 classification threshold documented in Assumptions.",
        "relevant": "Balances the underwriting-automation goal against real credit-risk exposure.",
        "timebound": "Target: reassess after the next full-data run confirms this ratio at scale.",
    },
]
if PD_INTEGRATION_AVAILABLE and pd_by_customer is not None and "UPSTREAM_PD_FROM_NB01" in NUMERIC_FEATURES:
    _shap_rank_ins = shap_importance_df.index[shap_importance_df["feature"] == "UPSTREAM_PD_FROM_NB01"].tolist()
    INSIGHTS.append({
        "headline": "This model now genuinely consumes Notebook 01's real default-risk signal",
        "specific": f"UPSTREAM_PD_FROM_NB01 (from Notebook 01's champion, {UPSTREAM_CHAMPION}) matched "
                    f"{N_MATCHED_TO_PD:,} / {N_JOINED:,} real in-scope rows ({N_MATCHED_TO_PD / N_JOINED:.2%}) "
                    + (f"and ranks #{_shap_rank_ins[0] + 1} of {len(FEATURE_COLS)} real features by mean |SHAP|."
                       if _shap_rank_ins else "."),
        "measurable": f"Champion mean CV ROC-AUC with this feature included: {cv_results[CHAMPION_NAME]['mean_auc']:.4f}.",
        "achievable": "Compare against a re-run with this feature excluded on the next full-data run to "
                      "quantify its real incremental lift.",
        "relevant": "A genuine, real cross-problem interdependency between Problem 1 (default prediction) and "
                    "Problem 3 (approval) -- subject to the explicit temporal-snapshot caveat above, which "
                    "must stay disclosed alongside this feature wherever it is used.",
        "timebound": "Target: re-verify after either notebook's underlying model or feature set changes.",
    })

insights_summary_df = pd.DataFrame(INSIGHTS)
_feature_source_names = list(CREDIT_HISTORY_FEATURES)
_feature_source_groups = (["bureau + bureau_balance"] * len(BUREAU_FEATURES)
                          + ["POS_CASH_balance / installments_payments / credit_card_balance (leave-one-out safe)"] * len(SERVICING_FEATURES))
if PD_INTEGRATION_AVAILABLE and pd_by_customer is not None and "UPSTREAM_PD_FROM_NB01" in NUMERIC_FEATURES:
    _feature_source_names = _feature_source_names + ["UPSTREAM_PD_FROM_NB01"]
    _feature_source_groups = _feature_source_groups + [
        "Notebook 01 (Problem 1) champion model -- real cross-problem interdependency, soft dependency"
    ]
credit_history_features_df = pd.DataFrame({
    "feature": _feature_source_names,
    "source_table_group": _feature_source_groups,
})

csv_paths = write_csv_outputs(
    {
        "notebook_02_holdout_predictions": holdout_predictions_df,
        "notebook_02_model_screening_top4": screening_df,
        "notebook_02_model_comparison": model_comparison_df,
        "notebook_02_cv_report_top2": cv_report_df,
        "notebook_02_shap_feature_importance": shap_importance_df,
        "notebook_02_lime_explanations": lime_explanations_df,
        "notebook_02_calibration_by_decile": calibration_df,
        "notebook_02_insights_summary": insights_summary_df,
        "notebook_02_credit_history_features": credit_history_features_df,
    },
    REPORTS_DIR,
)

word_sections = [
        {"heading": "Exploratory Data Analysis & Data Quality (SOP Stage 1B/2)",
         "paragraphs": [
             f"{len(null_counts)} of {df.width} real in-scope columns have at least one missing value "
             f"(top 5 by % missing shown in the chart below).",
             "IQR-based outlier counts (real): " + "; ".join(
                 f"{o['column']}={o['n_outliers_iqr']:,} ({o['pct_outliers_iqr']:.1%})" for o in OUTLIER_SUMMARY
             ) + ".",
             f"Real Pearson correlation with TARGET (top 3 by |r|): "
             f"{target_corr.abs().sort_values(ascending=False).head(3).round(4).to_dict()}.",
         ],
         "image_path": eda_overview_path,
         "story": STORY_MISSING_CHART},
        {"heading": "Numeric Distributions & Outliers", "image_path": eda_distributions_path},
        {"heading": "Correlation with Target", "image_path": eda_correlation_path},
        {"heading": "Multi-Table Credit-History Feature Integration",
         "paragraphs": [
             f"{len(BUREAU_FEATURES)} features from bureau + bureau_balance (external credit bureau, used in "
             f"full at applicant level — no linkage to any specific Home Credit decision).",
             f"{len(SERVICING_FEATURES)} features from POS_CASH_balance, installments_payments, and "
             f"credit_card_balance, computed as an exact leave-one-out aggregate (this applicant's OTHER "
             f"previous applications only) to guarantee zero leakage of the specific outcome being predicted.",
         ]},
        {"heading": "Cross-Notebook Feature: Real PD from Notebook 01 (soft dependency)"
                    + (f" — {UPSTREAM_CHAMPION}" if (PD_INTEGRATION_AVAILABLE and pd_by_customer is not None) else ""),
         "paragraphs": (
             [
                 f"{N_MATCHED_TO_PD:,} of {N_JOINED:,} real in-scope rows matched to a real customer PD from "
                 f"Notebook 01's champion model by SK_ID_CURR, added as a new real input feature, "
                 f"UPSTREAM_PD_FROM_NB01.",
                 "TEMPORAL-SNAPSHOT CAVEAT: this PD reflects each customer's CURRENT risk profile, not their "
                 "risk reconstructed at the actual historical moment of each specific previous-application "
                 "decision being predicted — a real, stated limitation, not leakage of this notebook's own "
                 "TARGET (a different real outcome column from Notebook 01's).",
             ] if (PD_INTEGRATION_AVAILABLE and pd_by_customer is not None) else STORY_INTERDEPENDENCY_CHART
         ),
         "story": STORY_INTERDEPENDENCY_CHART if (PD_INTEGRATION_AVAILABLE and pd_by_customer is not None) else None},
        {"heading": f"Stage A — Model Candidate Screening (top {len(CANDIDATE_NAMES)}, single real split)",
         "paragraphs": [
             f"All {len(CANDIDATE_NAMES)} real candidates ({', '.join(CANDIDATE_NAMES)}) were fit once on an "
             f"80/20 real train/validation split before committing any of them to the full 5-fold CV cost.",
         ],
         "table": {"headers": ["Model", "Screen ROC-AUC", "Fit Seconds", "Advanced to CV"],
                   "rows": [[n, f"{v['screen_auc']:.4f}", f"{v['fit_seconds']:.2f}", "Yes" if n in TOP2_NAMES else "No"]
                            for n, v in screening_results.items()]},
         "image_path": screening_chart_path},
        {"heading": "Stage B — Champion Selection (5-fold CV, top 2 real candidates only)",
         "table": {"headers": ["Model", "Mean CV ROC-AUC", "Std Dev"],
                   "rows": [[n, f"{cv_results[n]['mean_auc']:.4f}", f"{cv_results[n]['std_auc']:.4f}"] for n in cv_results]},
         "image_path": ARTIFACTS_DIR / "notebook_02_benchmark_chart.png",
         "story": STORY_MODEL_CHART},
        {"heading": "CV Report — Per-Fold Results (Top 2 Models)",
         "table": {"headers": ["Model", "Fold", "ROC-AUC"],
                   "rows": cv_report_df.values.tolist()},
         "image_path": cv_report_chart_path},
        {"heading": "Holdout Performance",
         "table": {"headers": ["Metric", "Value"],
                   "rows": [[k, f"{v:.4f}"] for k, v in metrics.items()]}},
        {"heading": f"SHAP Explainability (Champion Model: {CHAMPION_NAME})",
         "paragraphs": [
             f"Computed on a real {SHAP_SAMPLE_N}-row sample of the holdout set. Top 5 real drivers by mean "
             f"|SHAP value|: {TOP_SHAP_FEATURES.head(5)[['feature', 'mean_abs_shap']].round(4).to_dict('records')}.",
         ],
         "image_path": shap_summary_path,
         "story": STORY_SHAP_CHART},
        {"heading": "LIME Explainability (Champion Model, representative real holdout cases)",
         "paragraphs": [f"Real cases explained: {', '.join(_lime_cases.keys())}."],
         "table": {"headers": ["Case", "Feature Condition", "Weight"],
                   "rows": [[r["case"], r["feature_condition"], f"{r['weight']:.4f}"] for r in LIME_EXPLANATIONS]},
         "image_path": lime_panel_path,
         "story": STORY_LIME_CHART},
        {"heading": "Statistical Validation (SOP Stage 4)",
         "paragraphs": [
             f"Bootstrap 95% CI on holdout ROC-AUC ({N_BOOTSTRAP} resamples): [{AUC_CI_LOW:.4f}, {AUC_CI_HIGH:.4f}].",
             f"Mean calibration gap across {len(calib_summary)} score deciles: {MEAN_CALIBRATION_GAP:.4f} "
             f"(threshold: <{CALIBRATION_GAP_THRESHOLD}).",
             f"Split-half PSI on holdout score distribution: {SPLIT_HALF_PSI:.4f} (threshold: <{PSI_STABILITY_THRESHOLD}).",
         ],
         "story": STORY_CALIB_CHART},
        {"heading": "Financial Impact (real volume figures + one labeled assumption)",
         "paragraphs": [
             f"Total real holdout requested credit volume (AMT_CREDIT): ${TOTAL_HOLDOUT_VOLUME:,.0f}.",
             f"Volume of correctly-identified real approvals (true positives): ${CORRECTLY_APPROVED_VOLUME:,.0f}.",
             f"Volume of missed real approvals (false negatives): ${MISSED_APPROVAL_VOLUME:,.0f}.",
             f"Volume wrongly flagged approvable — real declines incorrectly flagged (false positives): ${WRONGLY_APPROVED_VOLUME:,.0f}.",
             f"ASSUMPTION (illustrative manual-review cost, ${ASSUMPTIONS['AVG_PROCESSING_COST_PER_MANUAL_REVIEW']:.0f}/review): "
             f"applied only to correctly-identified approvals, illustrative manual-review cost avoided = "
             f"${ESTIMATED_MANUAL_REVIEW_COST_AVOIDED:,.0f}. This is a labeled estimate, not a real observed figure.",
         ],
         "story": STORY_VOLUME_CHART},
        {"heading": "Integrity Checks",
         "table": {"headers": ["Check", "Result"],
                   "rows": [[name, "PASS" if ok else "FAIL"] for name, ok in checks]}},
]

word_path = build_word_report(
    REPORTS_DIR / "notebook_02_report.docx",
    title="Problem 3 — Loan Application Approval",
    subtitle="Mega Project 1: Intelligent Underwriting & Automated Credit Decisioning",
    exec_summary=[
        f"Champion model: {CHAMPION_NAME}, selected by highest mean 5-fold CV ROC-AUC among the top 2 of "
        f"{len(CANDIDATE_NAMES)} real screened candidates ({RUNNER_UP_NAME} runner-up).",
        f"Real holdout ROC-AUC: {metrics['roc_auc']:.4f} (95% bootstrap CI [{AUC_CI_LOW:.4f}, {AUC_CI_HIGH:.4f}]).",
        f"Deployment readiness verdict: {DEPLOYMENT_VERDICT}",
        f"{len(CREDIT_HISTORY_FEATURES)} real applicant credit-history features integrated from 5 additional "
        f"Home Credit tables, for {len(FEATURE_COLS)} total real features.",
        (f"UPSTREAM_PD_FROM_NB01 (real PD from Notebook 01's champion, {UPSTREAM_CHAMPION}) included as a "
         f"real input feature, subject to the disclosed temporal-snapshot caveat."
         if (PD_INTEGRATION_AVAILABLE and pd_by_customer is not None) else
         "Notebook 01's PD feature not included this run (Notebook 01 not yet run) — soft dependency."),
        f"SHAP + LIME explainability computed for the champion model only; top real driver: "
        f"{TOP_SHAP_FEATURES.iloc[0]['feature']}.",
        f"All {len(checks)} pipeline integrity checks: {sum(1 for _, ok in checks if ok)}/{len(checks)} PASS.",
    ],
    insights=INSIGHTS,
    sections=word_sections,
)

review_cost_ref = assumption_ref(ASSUMPTIONS, "AVG_PROCESSING_COST_PER_MANUAL_REVIEW")
excel_path = build_excel_workbook(
    REPORTS_DIR / "notebook_02_workbook.xlsx",
    assumptions=ASSUMPTIONS,
    assumption_notes=ASSUMPTION_NOTES,
    data_sheets=[
        {"name": "Model Screening (Top 4)", "headers": ["Model", "Screen ROC-AUC", "Fit Seconds", "Advanced to CV"],
         "rows": [[n, v["screen_auc"], v["fit_seconds"], "Yes" if n in TOP2_NAMES else "No"]
                  for n, v in screening_results.items()],
         "highlight_col": "Screen ROC-AUC"},
        {"name": "Model Comparison", "headers": ["Model", "Mean CV AUC", "Std Dev"],
         "rows": [[n, cv_results[n]["mean_auc"], cv_results[n]["std_auc"]] for n in cv_results],
         "highlight_col": "Mean CV AUC"},
        {"name": "CV Report (Top 2, 5-fold)", "headers": ["Model", "Fold", "ROC-AUC"],
         "rows": cv_report_df.values.tolist(), "highlight_col": "ROC-AUC"},
        {"name": "Holdout Metrics", "headers": ["Metric", "Value"], "rows": [[k, v] for k, v in metrics.items()]},
        {"name": "Calibration by Decile", "headers": ["Score Decile", "Mean Predicted", "Actual Rate", "N"],
         "rows": calibration_df.values.tolist(), "highlight_col": "Actual Rate"},
        {"name": "SHAP Feature Importance", "headers": ["Feature", "Mean |SHAP|"],
         "rows": shap_importance_df.values.tolist(), "highlight_col": "Mean |SHAP|"},
        {"name": "LIME Instance Explanations", "headers": ["Case", "Feature Condition", "Weight"],
         "rows": [[r["case"], r["feature_condition"], r["weight"]] for r in LIME_EXPLANATIONS]},
        {"name": "Credit History Features", "headers": ["Feature", "Source Table Group"],
         "rows": credit_history_features_df.values.tolist()},
        {"name": "Integrity Checks", "headers": ["Check", "Result"],
         "rows": [[name, "PASS" if ok else "FAIL"] for name, ok in checks]},
    ],
    formula_sheet={
        "name": "Financial Impact",
        "rows": [
            ("Total Holdout Volume ($)", TOTAL_HOLDOUT_VOLUME),
            ("Correctly Approved Volume ($, TP)", CORRECTLY_APPROVED_VOLUME),
            ("Missed Approval Volume ($, FN)", MISSED_APPROVAL_VOLUME),
            ("Wrongly Flagged Approve Volume ($, FP)", WRONGLY_APPROVED_VOLUME),
            ("Capture Rate", CAPTURE_RATE),
            ("Est. Manual-Review Cost Avoided ($, illustrative)", f"={int(tp_n)}*{review_cost_ref}"),
        ],
    },
    insights_sheet={"name": "Insights & SMART Actions", "items": INSIGHTS},
)

# --- Real alternate "slicer" views: same underlying real numbers, sliced a
# second honest way, so the dashboard's filter dropdowns switch between real
# precomputed data rather than fabricating anything client-side. ------------
model_view_cv = {"key": "cv", "label": "Top-2 5-Fold CV (Champion Selection)",
                  "labels": list(cv_results.keys()),
                  "datasets": [{"label": "Mean CV ROC-AUC", "data": [cv_results[n]["mean_auc"] for n in cv_results],
                                "backgroundColor": [VIVID_PALETTE[1] if n == CHAMPION_NAME else VIVID_PALETTE[4]
                                                     for n in cv_results]}]}
model_view_screen = {"key": "screen", "label": f"Stage A Screening (All {len(CANDIDATE_NAMES)} Candidates)",
                      "labels": list(screening_results.keys()),
                      "datasets": [{"label": "Single-Split Screen AUC",
                                    "data": [v["screen_auc"] for v in screening_results.values()],
                                    "backgroundColor": [VIVID_PALETTE[1] if n in TOP2_NAMES else "#a9a9a9"
                                                         for n in screening_results]}]}

cv_report_view = {"key": "cvreport", "label": "5-Fold CV Report",
                   "labels": [f"Fold {i}" for i in range(1, 6)],
                   "datasets": [
                       {"label": name, "data": cv_results[name]["fold_aucs"],
                        "backgroundColor": VIVID_PALETTE[1] if name == CHAMPION_NAME else VIVID_PALETTE[4]}
                       for name in TOP2_NAMES
                   ]}

shap_view = {"key": "shap", "label": "SHAP Feature Importance",
             "labels": TOP_SHAP_FEATURES["feature"].tolist(),
             "datasets": [{"label": "Mean |SHAP|", "data": TOP_SHAP_FEATURES["mean_abs_shap"].round(4).tolist(),
                           "backgroundColor": _palette(len(TOP_SHAP_FEATURES))}]}

_lime_first_case = next(iter(_lime_cases.keys())) if _lime_cases else None
_lime_first_rows = [r for r in LIME_EXPLANATIONS if r["case"] == _lime_first_case] if _lime_first_case else []
lime_view = {"key": "lime", "label": f"LIME — {_lime_first_case}" if _lime_first_case else "LIME",
             "labels": [r["feature_condition"] for r in _lime_first_rows],
             "datasets": [{"label": "Local Weight", "data": [round(r["weight"], 4) for r in _lime_first_rows],
                           "backgroundColor": [VIVID_PALETTE[2] if r["weight"] >= 0 else VIVID_PALETTE[7]
                                                for r in _lime_first_rows]}]}

top_missing_5 = top_missing.head(5)
missing_view_15 = {"key": "top15", "label": f"Top {len(top_missing)} Columns",
                    "labels": top_missing["column"].tolist(),
                    "datasets": [{"label": "% Missing", "data": (top_missing["pct_null"] * 100).round(2).tolist(),
                                  "backgroundColor": _palette(len(top_missing))}]}
missing_view_5 = {"key": "top5", "label": "Top 5 Columns",
                   "labels": top_missing_5["column"].tolist(),
                   "datasets": [{"label": "% Missing", "data": (top_missing_5["pct_null"] * 100).round(2).tolist(),
                                 "backgroundColor": _palette(len(top_missing_5))}]}

volume_view_dollars = {"key": "dollars", "label": "By Dollar Volume",
                        "labels": ["Correctly Approved (TP)", "Missed Approval (FN)", "Wrongly Flagged (FP)"],
                        "datasets": [{"data": [CORRECTLY_APPROVED_VOLUME, MISSED_APPROVAL_VOLUME, WRONGLY_APPROVED_VOLUME],
                                      "backgroundColor": [VIVID_PALETTE[0], VIVID_PALETTE[7], VIVID_PALETTE[3]]}]}
volume_view_count = {"key": "count", "label": "By Application Count",
                      "labels": ["Correctly Approved (TP)", "Missed Approval (FN)", "Wrongly Flagged (FP)"],
                      "datasets": [{"data": [int(tp_n), int(fn_n), int(fp_n)],
                                    "backgroundColor": [VIVID_PALETTE[0], VIVID_PALETTE[7], VIVID_PALETTE[3]]}]}

SAMPLE_N = min(200, len(holdout_predictions_df))
sample_df = (
    holdout_predictions_df.sample(n=SAMPLE_N, random_state=SEED)
    .sort_values("predicted_probability", ascending=False)
    .round({"predicted_probability": 4, "amt_credit": 2})
)

html_path = build_html_dashboard(
    REPORTS_DIR / "notebook_02_dashboard.html",
    title="Problem 3 — Loan Application Approval",
    subtitle=f"Champion: {CHAMPION_NAME} | Real holdout ROC-AUC: {metrics['roc_auc']:.4f} | {DEPLOYMENT_VERDICT}",
    kpi_cards=[
        {"label": "Champion Model", "value": CHAMPION_NAME},
        {"label": "Holdout ROC-AUC", "value": f"{metrics['roc_auc']:.4f}"},
        {"label": "Capture Rate", "value": f"{CAPTURE_RATE:.1%}"},
        {"label": "Correctly Approved Volume", "value": f"${CORRECTLY_APPROVED_VOLUME:,.0f}"},
        {"label": "Real Approval Rate", "value": f"{APPROVAL_RATE:.2%}"},
        {"label": "Credit-History Features", "value": f"{len(CREDIT_HISTORY_FEATURES)} added"},
        {"label": "Top SHAP Driver", "value": TOP_SHAP_FEATURES.iloc[0]["feature"]},
        {"label": "NB01 PD Feature",
         "value": "Included" if (PD_INTEGRATION_AVAILABLE and pd_by_customer is not None) else "N/A (run NB01)"},
    ],
    insights=INSIGHTS,
    charts=[
        {"id": "modelChart", "title": "Top-2 5-Fold CV Champion Selection", "type": "bar",
         "labels": model_view_cv["labels"], "datasets": model_view_cv["datasets"], "showLegend": False,
         "views": [model_view_cv, model_view_screen], "story": STORY_MODEL_CHART},
        {"id": "cvReportChart", "title": "5-Fold CV Report (Top 2 Models, Per-Fold Real ROC-AUC)", "type": "bar",
         "labels": cv_report_view["labels"], "datasets": cv_report_view["datasets"], "showLegend": True},
        {"id": "shapChart", "title": f"SHAP Feature Importance — Champion ({CHAMPION_NAME})", "type": "bar",
         "labels": shap_view["labels"], "datasets": shap_view["datasets"], "showLegend": False,
         "story": STORY_SHAP_CHART},
        {"id": "limeChart", "title": f"LIME — {_lime_first_case or 'Champion'} (real holdout case)", "type": "bar",
         "labels": lime_view["labels"], "datasets": lime_view["datasets"], "showLegend": False,
         "story": STORY_LIME_CHART},
        {"id": "missingChart", "title": "Top Columns by Real Missing-Value %", "type": "bar",
         "labels": missing_view_15["labels"], "datasets": missing_view_15["datasets"], "showLegend": False,
         "views": [missing_view_15, missing_view_5], "story": STORY_MISSING_CHART},
        {"id": "calibChart", "title": "Calibration by Score Decile (real holdout)", "type": "line",
         "labels": [str(int(d)) for d in calib_summary["decile"]],
         "datasets": [
             {"label": "Mean Predicted", "data": calib_summary["mean_predicted"].round(4).tolist(), "borderColor": VIVID_PALETTE[0], "backgroundColor": VIVID_PALETTE[0], "fill": False},
             {"label": "Actual Approval Rate", "data": calib_summary["actual_rate"].round(4).tolist(), "borderColor": VIVID_PALETTE[7], "backgroundColor": VIVID_PALETTE[7], "fill": False},
         ],
         "note": f"Mean calibration gap: {MEAN_CALIBRATION_GAP:.4f}", "story": STORY_CALIB_CHART},
        {"id": "volumeChart", "title": "Holdout Credit Volume by Outcome (real AMT_CREDIT)", "type": "doughnut",
         "labels": volume_view_dollars["labels"], "datasets": volume_view_dollars["datasets"],
         "views": [volume_view_dollars, volume_view_count], "story": STORY_VOLUME_CHART},
        {"id": "targetChart", "title": "Real Approval Outcome Balance", "type": "doughnut",
         "labels": ["Not Approved (0)", "Approved (1)"],
         "datasets": [{"data": target_counts["count"].tolist(), "backgroundColor": ["#d03b3b", "#0ca30c"]}],
         "story": STORY_TARGET_CHART},
    ],
    data_table={
        "title": f"Sampled Real Holdout Predictions ({SAMPLE_N} of {len(holdout_predictions_df):,} rows)",
        "columns": ["SK_ID_PREV", "actual_target", "predicted_probability", "predicted_label", "amt_credit", "outcome"],
        "rows": sample_df[["SK_ID_PREV", "actual_target", "predicted_probability", "predicted_label", "amt_credit", "outcome"]].values.tolist(),
        "filter_column": "outcome",
    },
)
print(f"[REPORTING] Real reporting package written: reports/{word_path.name}, reports/{excel_path.name}, "
      f"reports/{html_path.name}, plus {len(csv_paths)} CSV file(s) (all under decision_engine/reports/).")

# ---------------------------------------------------------------------------
# SECTION 15 — Save artifacts + governance stamp (SOP Stage 6: Production
# Packaging & Governance) — idempotent: overwrite in place, fixed paths
# ---------------------------------------------------------------------------
summary = {
    "notebook": "02_loan_application_approval",
    "mega_project": "Mega Project 1 - Intelligent Underwriting & Automated Credit Decisioning",
    "problem": "Problem 3 - Loan Application Approval",
    "random_seed": SEED,
    "n_rows_previous_application_raw": N_PREV_RAW,
    "n_rows_in_scope": N_JOINED,
    "join_scope_rate": join_rate,
    "feature_count": len(FEATURE_COLS),
    "credit_history_feature_count": len(CREDIT_HISTORY_FEATURES),
    "raw_tables_integrated": REQUIRED_FILES,
    "leakage_columns_excluded": sorted(LEAKAGE_COLS),
    "eda_data_quality": {
        "n_columns_with_missing": int(len(null_counts)),
        "top_5_missing_pct": {row["column"]: round(float(row["pct_null"]), 4) for _, row in null_counts.head(5).iterrows()},
        "iqr_outliers": OUTLIER_SUMMARY,
        "top_target_correlations": target_corr.abs().sort_values(ascending=False).head(3).round(4).to_dict(),
        "eda_chart_files": [p.name for p in EDA_CHART_PATHS],
    },
    "performance_config": {
        "logical_cores_detected": TOTAL_THREADS,
        "total_ram_gb_detected": TOTAL_RAM_GB,
        "cpu_thread_ceiling_applied": CPU_CEILING_THREADS,
        "ram_ceiling_gb": RAM_CEILING_GB,
        "cpu_affinity_pinned_cores": PERF.get("logical_cores"),
        "parquet_cache_dir": str(PARQUET_CACHE_DIR.name),
    },
    "model_selection": {
        "candidate_models_screened": CANDIDATE_NAMES,
        "screening_results": screening_results,
        "top2_advanced_to_cv": TOP2_NAMES,
        "champion_model": CHAMPION_NAME,
        "runner_up_model": RUNNER_UP_NAME,
        "cv_results": cv_results,
    },
    "explainability": {
        "shap_sample_size": SHAP_SAMPLE_N,
        "shap_top_10_features": shap_importance_df.head(10).round(4).to_dict("records"),
        "lime_cases_explained": list(_lime_cases.keys()),
        "lime_explanation_count": len(LIME_EXPLANATIONS),
    },
    "interdependency_with_notebook_01": (
        {
            "available": True,
            "feature_name": "UPSTREAM_PD_FROM_NB01",
            "upstream_champion": UPSTREAM_CHAMPION,
            "n_matched_rows": N_MATCHED_TO_PD,
            "pct_matched_rows": N_MATCHED_TO_PD / N_JOINED if N_JOINED else 0.0,
            "temporal_snapshot_caveat": "PD reflects each customer's CURRENT risk profile (application_train), "
                                        "not a reconstruction of risk at the historical moment of each specific "
                                        "previous-application decision being predicted -- a disclosed limitation, "
                                        "not leakage of this notebook's own TARGET.",
        } if (PD_INTEGRATION_AVAILABLE and pd_by_customer is not None) else
        {"available": False, "reason": "Notebook 01 has not been run yet on this machine (soft dependency)."}
    ),
    "cv_results": cv_results,
    "champion_model": CHAMPION_NAME,
    "holdout_metrics": metrics,
    "statistical_validation": {
        "bootstrap_resamples": N_BOOTSTRAP,
        "holdout_auc_ci_95": [AUC_CI_LOW, AUC_CI_HIGH],
        "mean_calibration_gap": MEAN_CALIBRATION_GAP,
        "split_half_psi": SPLIT_HALF_PSI,
        "deployment_checks": {name: ok for name, ok in deployment_checks},
        "failed_deployment_checks": _failed_deployment_checks,
        "deployment_verdict": DEPLOYMENT_VERDICT,
        "note": "deployment_checks (statistical robustness) is a separate check family from "
                "integrity_checks (structural pipeline sanity) below -- see deployment_verdict "
                "for which specific statistical check(s), if any, failed on this run.",
    },
    "financial_impact": {
        "total_holdout_volume_usd": TOTAL_HOLDOUT_VOLUME,
        "correctly_approved_volume_usd": CORRECTLY_APPROVED_VOLUME,
        "missed_approval_volume_usd": MISSED_APPROVAL_VOLUME,
        "wrongly_approved_volume_usd": WRONGLY_APPROVED_VOLUME,
        "capture_rate": CAPTURE_RATE,
        "assumptions": ASSUMPTIONS,
        "estimated_manual_review_cost_avoided_usd_illustrative": ESTIMATED_MANUAL_REVIEW_COST_AVOIDED,
    },
    "integrity_checks": {name: bool(ok) for name, ok in checks},
    "reporting_artifacts": ["notebook_02_report.docx", "notebook_02_workbook.xlsx",
                             "notebook_02_dashboard.html"] + [f"{stem}.csv" for stem in csv_paths],
    "sop_stage_reached": "6 - Production Packaging & Governance",
    "runtime_seconds": round(time.time() - T0, 1),
}
with open(ARTIFACTS_DIR / "notebook_02_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

joblib.dump(
    {"model": champion, "ordinal_encoder": ord_enc, "imputer": imputer, "feature_cols": FEATURE_COLS,
     "numeric_features": NUMERIC_FEATURES, "categorical_features": CATEGORICAL_FEATURES,
     "champion_name": CHAMPION_NAME},
    ARTIFACTS_DIR / "notebook_02_champion_model.joblib",
)
print(f"[DONE] Notebook 02 complete in {summary['runtime_seconds']}s using a {CPU_CEILING_THREADS}-thread "
      f"WARP ceiling. Champion={CHAMPION_NAME} (of top-2 {TOP2_NAMES} from a {len(CANDIDATE_NAMES)}-model "
      f"screen), holdout ROC-AUC={metrics['roc_auc']:.4f}. "
      f"Deployment verdict: {DEPLOYMENT_VERDICT}. "
      f"{len(CREDIT_HISTORY_FEATURES)} credit-history features integrated from 5 additional real tables.")
